In [15]:
# Add project root to Python path
import sys
from pathlib import Path

# Get the project root (notebook is in akd/agents/story_teller/)
project_root = Path.cwd()
if 'story_teller' in str(project_root):
    # Navigate up to project root
    project_root = project_root.parent.parent.parent
    
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path includes project root: {str(project_root) in sys.path}")

Project root: /Users/thapa24-mbp/Devs/sandbox/accelerated-discovery
Python path includes project root: True


In [ ]:

"""
Embedded instruction builder component for story teller agents.
"""

from typing import List, Optional

from loguru import logger
from pydantic import Field

from akd._base import InputSchema, OutputSchema
from akd.agents._base import BaseAgentConfig, LiteLLMInstructorBaseAgent
from akd.configs.prompts import STORY_TELLER_AGENT_PROMPT


class StoryMDXBuilderInputSchema(InputSchema):
    """Input schema for story mdx builder component."""

    query: str = Field(..., description="A Query to build MDX story out of.")
    
    # datasets: List[List[datasetId, description, List[List[LayerId, description]]]] = Field(
    datasets: List[str] = Field(
      default=None,
      description="This is the list of relevant dataset which is available in STAC. This contains the datasetId, its description, and a list of layerIds for that datasetId."
    ) # or have a dict
    raw_contents: str = Field(
      default=None,
      description="This is the relevant data which is scraped from relevant webpages, pdfs and more."
    ) # maybe use a rag based store for references. this is all the datasets that came from pdfs, links or others, provided by LiteratureSearchAgent


class StoryMDXBuilderOutputSchema(OutputSchema):
    """Output schema for story mdx builder component."""

    story: str = Field(
      ...,
      description="A story written in mdx with veda-ui blocks."
    )


class StoryMDXBuilderConfig(BaseAgentConfig):
    """Configuration for the story mdx builder component."""

    system_prompt: str = STORY_TELLER_AGENT_PROMPT
    model_name: str = "gpt-4o-mini"
    temperature: float = 0.3


class StoryMDXBuilderComponent:
    """
    Story MDX builder component that creates a story driven detailed MDX.

    This component is embedded within Story teller Agent to provide
    MDX building functionality without requiring separate agent instantiation.
    """

    def __init__(
        self,
        config: Optional[StoryMDXBuilderConfig] = None,
        debug: bool = False,
    ):
        self.config = config or StoryMDXBuilderConfig()
        self.debug = debug

        # Create internal MDX generation agent for instruction building
        self._agent = LiteLLMInstructorBaseAgent[
            StoryMDXBuilderInputSchema,
            StoryMDXBuilderOutputSchema,
        ](config=self.config, debug=debug)
        self._agent.input_schema = StoryMDXBuilderInputSchema
        self._agent.output_schema = StoryMDXBuilderOutputSchema

    async def process(
        self,
        query: str,
        datasets: List[str] = None,
        raw_contents: str = None
    ) -> str:
        """
        Build detailed MDX output from query, available datasets and raw_contents.

        Args:
            query: A Query to build MDX story out of.
            datasets: The list of relevant dataset which is available in STAC. This contains the datasetId, its description, and a list of layerIds for that datasetId.
            raw_contents: The relevant data which is scraped from relevant webpages, pdfs and more.

        Returns:
            A story written in mdx with veda-ui blocks.
        """
        if self.debug:
            logger.debug(f"Building story MDX generation for query: {query[:100]}...")

        input = StoryMDXBuilderInputSchema(
            query=query,
            datasets=datasets,
            raw_contents=raw_contents
        )

        output = await self._agent.arun(input)

        if self.debug:
            logger.debug(
                f"Generated Story MDX string of ({len(output.story)} chars)",
            )
            logger.debug(f"Story MDX: {output.story}")

        return output.story


In [ ]:
query=""

In [ ]:
datasets=[]
raw_contents="""
Atmos. Chem. Phys., 22, 9617–9646, 2022
https://doi.org/10.5194/acp-22-9617-2022
© Author(s) 2022. This work is distributed under
the Creative Commons Attribution 4.0 License.
Review article
Quantifying methane emissions from the global scale
down to point sources using satellite observations
of atmospheric methane
Daniel J. Jacob1
, Daniel J. Varon1,2
, Daniel H. Cusworth3,4
, Philip E. Dennison5
,
Christian Frankenberg6,7
, Ritesh Gautam8
, Luis Guanter9,10, John Kelley11, Jason McKeever2
,
Lesley E. Ott12, Benjamin Poulter12, Zhen Qu1
, Andrew K. Thorpe7
, John R. Worden7
, and
Riley M. Duren3,4,7
1School of Engineering and Applied Sciences, Harvard University, Cambridge, 02138, USA
2GHGSat, Inc., Montreal, H2W 1Y5, Canada
3Arizona Institutes for Resilience, University of Arizona, Tucson, 85721, USA
4Carbon Mapper, Pasadena, 91109, USA
5Department of Geography, University of Utah, Salt Lake City, 84112, USA
6Division of Geological and Planetary Sciences, California Institute of Technology, Pasadena, 91125, USA
7
Jet Propulsion Laboratory, California Institute of Technology, Pasadena, 91109, USA
8Environmental Defense Fund, Washington, D.C., 20009, USA
9Research Institute of Water and Environmental Engineering,
Universitat Politecnica de Valencia, Valencia, 46022, Spain
10Environmental Defense Fund, Amsterdam, 1017, The Netherlands
11GeoSapient, Inc., Cypress, 77429, USA
12NASA GSFC, Greenbelt, 20771, USA
Correspondence: Daniel J. Jacob (djacob@fas.harvard.edu)
Received: 1 April 2022 – Discussion started: 11 April 2022
Revised: 18 June 2022 – Accepted: 22 June 2022 – Published: 29 July 2022
Abstract. We review the capability of current and scheduled satellite observations of atmospheric methane in
the shortwave infrared (SWIR) to quantify methane emissions from the global scale down to point sources. We
cover retrieval methods, precision and accuracy requirements, inverse and mass balance methods for inferring
emissions, source detection thresholds, and observing system completeness. We classify satellite instruments as
area flux mappers and point source imagers, with complementary attributes. Area flux mappers are high-precision
(< 1 %) instruments with 0.1–10 km pixel size designed to quantify total methane emissions on regional to
global scales. Point source imagers are fine-pixel (< 60 m) instruments designed to quantify individual point
sources by imaging of the plumes. Current area flux mappers include GOSAT (2009–present), which provides
a high-quality record for interpretation of long-term methane trends, and TROPOMI (2018–present), which
provides global continuous daily mapping to quantify emissions on regional scales. These instruments already
provide a powerful resource to quantify national methane emissions in support of the Paris Agreement. Current
point source imagers include the GHGSat constellation and several hyperspectral and multispectral land imaging
sensors (PRISMA, Sentinel-2, Landsat-8/9, WorldView-3), with detection thresholds in the 100–10 000 kg h−1
range that enable monitoring of large point sources. Future area flux mappers, including MethaneSAT, GOSATGW, Sentinel-5, GeoCarb, and CO2M, will increase the capability to quantify emissions at high resolution, and
the MERLIN lidar will improve observation of the Arctic. The averaging times required by area flux mappers to
quantify regional emissions depend on pixel size, retrieval precision, observation density, fraction of successful
retrievals, and return times in a way that varies with the spatial resolution desired. A similar interplay applies
to point source imagers between detection threshold, spatial coverage, and return time, defining an observing
Published by Copernicus Publications on behalf of the European Geosciences Union.
9618 D. J. Jacob et al.: Quantifying methane emissions using satellites
system completeness. Expanding constellations of point source imagers including GHGSat and Carbon Mapper
over the coming years will greatly improve observing system completeness for point sources through dense
spatial coverage and frequent return times.
1 Introduction
Methane is a powerful greenhouse gas that has contributed
0.6 ◦C of global warming since pre-industrial times (Naik et
al., 2021). It is emitted by a number of anthropogenic source
sectors, including livestock, oil and gas systems, coal mining, landfills, wastewater treatment, and rice cultivation. Wetlands are the main natural source. The main sink is oxidation
by the hydroxyl radical (OH), resulting in an atmospheric
lifetime of about 9 years (Prather et al., 2012). Because of
this short lifetime, decreasing methane emissions is a powerful lever to slow down near-term greenhouse warming (Nisbet et al., 2020). However, methane emission estimates and
the contributions from different sectors are highly uncertain
(Saunois et al., 2020), hindering climate policy. Here we review the capability of satellite observations of atmospheric
methane to quantify emissions from the global scale down to
point sources.
Methane emission inventories are typically constructed using bottom-up methods in which activity levels (such as number of cows) are multiplied by emission factors (methane
emitted per cow; IPCC, 2019). Bottom-up methods relate
emissions to the underlying processes, thus providing a basis for emission control strategies. Observations of atmospheric methane provide top-down information to improve
these emission estimates by using inverse methods to relate
observed concentrations to emissions (Miller and Michalak,
2017). Satellite observations are of particular interest for this
purpose because of their high observation density and global
coverage (Palmer et al., 2021).
Satellites retrieve atmospheric methane column concentrations with near-unit sensitivity down to the surface by measuring spectrally resolved backscattered solar radiation in the
shortwave infrared (SWIR; Jacob et al., 2016). Global observation of methane from space began with the SCIAMACHY
instrument (2003–2014, 30×60 km2 pixels) (Frankenberg et
al., 2005) and has continued since with the TANSO-FTS instrument aboard GOSAT (2009–present, 10 km circular pixels separated by about 270 km; Parker et al., 2020) and
the TROPOMI instrument (2018–present, 5.5 × 7 km2 pixels; Lorente et al., 2021a). Many studies have used these
satellite observations to quantify methane emissions globally
(Bergamaschi et al., 2013; Alexe et al., 2015; Wang et al.,
2019; Qu et al., 2021), on continental scales (Wecht et al.,
2014; Maasakkers et al., 2021; Lu et al., 2022), on finer regional scales (Miller et al., 2019; Zhang et al., 2020; Shen et
al., 2021), and for large point sources (Pandey et al., 2019;
Sadavarte et al., 2021; Lauvaux et al., 2022; Maasakkers et
al., 2022a, b). Targeted observation of methane point sources
from space began with the 2015 Aliso Canyon blowout using the Hyperion hyperspectral sensor (Thompson et al.,
2016) and has since continued with the GHGSat instruments (2016–present, 25 × 25 m2 pixels; Jervis et al., 2021).
Hyperspectral land-imaging spectrometers (measuring continuous spectra with ∼ 10 nm resolution in selected wavelength channels) and multispectral land-imaging spectrometers (measuring radiances in discrete ∼ 100 nm channels)
have also demonstrated capability to detect large methane
point sources in their SWIR bands (Cusworth et al., 2019;
Guanter et al., 2021; Varon et al., 2021; Ehret et al., 2022;
Sanchez-Garcia et al., 2022).
Better quantification of methane emissions worldwide is
urgently needed to meet the demands of climate policy. Individual countries must report their emissions by sector to the
United Nations Framework Convention on Climate Change
(UNFCCC) on a yearly basis for Annex I (developed) countries. The enhanced transparency framework of the Paris
Agreement requires all countries to submit national sectorresolved emissions for expert review by November 2024 as
basis for setting their nationally determined contributions to
meet climate goals. Independently of the Paris Agreement,
over 110 countries have now signed the Global Methane
Pledge of 2021, committing them to reduce their collective
2030 methane emissions by 30 % relative to 2020 levels.
Satellites can help to quantify national emissions by sector
as baseline for setting methane reduction goals and can then
monitor emissions over time to evaluate success in achieving those goals. They provide near-real-time information on
emissions, whereas bottom-up inventories typically have latencies of a few years, and are thus a unique resource to document rapid changes in emissions (Barré et al., 2021).
Jacob et al. (2016) previously reviewed the state of the science for quantifying methane emissions from space. They
presented observing capabilities at the time, discussed the
inverse methods for inferring methane emissions from satellite observations, and laid out observing requirements for future satellite missions. Since then, new satellite instruments
for measuring atmospheric methane have been launched and
new capabilities for detecting methane point sources from
space have emerged. New analytical tools have been developed to infer emissions from satellite observations, including
for point sources. Additional satellite instruments are scheduled to be launched over the next few years that will augment
current capabilities. These new developments motivate our
updated review.
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9619
2 Observing atmospheric methane from space
2.1 Current and planned instruments
Table 1 lists current and scheduled satellite instruments with
documented or expected capability for quantifying methane
emissions, and Table 2 gives specific attributes for each. We
classify the instruments as area flux mappers or point source
imagers, and Fig. 1 illustrates these two fleets. Area flux
mappers are designed to observe total emissions on global or
regional scales with 0.1–10 km pixel size. Point source imagers are fine-pixel (< 60 m) instruments designed to quantify individual point sources by imaging the plumes. Point
source imagers have much finer spatial resolution than area
flux mappers but lower precision.
All instruments in Table 1 except MERLIN observe
methane by SWIR solar backscatter from the Earth’s surface, either at 1.63–1.70 µm (1.65 µm band) or at 2.2–2.4 µm
(2.3 µm band). Atmospheric scattering is weak in the SWIR
except for clouds and large aerosol particles. Under clear
skies, methane is observed down to the surface with near
unit sensitivity (Worden et al., 2015). The retrieval may fail if
the surface is too dark, such as over water or forest canopies
(Ayasse et al., 2018). Observations over water can be made
by sunglint when the sun–satellite viewing geometry is favorable. The MERLIN lidar instrument emits its own 1.65 µm
radiation and detects the reflected signal. It can observe over
water and at night, but its sensitivity and coverage are lower
than for the solar backscatter instruments. Lidar capability
to observe methane from space is currently limited by laser
technology (Riris et al., 2019).
Not included in Table 1 are instruments that measure
methane in the thermal infrared (TIR) or by solar occultation. These instruments are not sensitive to methane near
the surface and are therefore not directly useful for quantifying methane emissions. TIR instruments have been used
for remote sensing of methane plumes from aircraft (Hulley
et al., 2016), but measurements from satellites mainly sense
the upper tropospheric background (Worden et al., 2015). Solar occultation instruments such as ACE-FTS provide sensitive measurements of stratospheric methane profiles (Koo et
al., 2017) but cloud interference prevents observations in the
troposphere. TIR and solar occultation instruments can complement SWIR data by providing information on background
methane in the upper troposphere and stratosphere (Zhang et
al., 2021; Tu et al., 2022).
The spectrally resolved SWIR backscattered solar radiation detected by satellite under clear-sky conditions can be
used to retrieve the total atmospheric column of methane,
CH4
[molecules cm−2
], as will be reviewed in Sect. 2.2.
To remove the variability from surface pressure, measurements are typically reported as dry column mixing ratio XCH4 = CH4
/a,d, where a,d is the dry air column
[molecules cm−2
]. Normalizing to dry air rather than total
air avoids introducing dependence on water vapor.
All instruments in Table 1 except EMIT and GeoCarb are
in low-elevation polar sun-synchronous orbit and observe
globally at specific local times of day, either morning or early
afternoon. Morning has greater probability of clear sky, while
early afternoon has steadier boundary layer winds for interpreting methane enhancements. GOSAT (2009–present) and
its follow-on GOSAT-2 (2018–present) provide global coverage every 3 d for 10 km circular pixels spaced about 270 km
apart, while TROPOMI (2018–present) provides full global
daily coverage with 5.5×7 km2 pixels. Figure 2 shows mean
TROPOMI XCH4
data for two different seasons, illustrating
the dense coverage. Future instruments GOSAT-GW (2023
launch, 10×10 km2 pixels with full global coverage every 3 d
in wide-swath mode), Sentinel-5 (2024 launch, 7.5×7.5 km2
pixels with full global daily coverage), and CO2M (2025
launch, 2 × 2 km2 pixels with full global coverage every 5 d)
will continue the global observation record. MERLIN will
provide day and night global coverage along its lidar orbit
track. Sentinel-2 and Landsat instruments provide full global
coverage with 20–30 m pixels every 5 d (Sentinel-2) or 16 d
(Landsat) and can detect very large point sources over bright
spectrally homogeneous surfaces. EMIT (designed to observe arid surfaces for dust generation) will be on a 51.6◦
inclined orbit aboard the International Space Station with variable local overpass times. GeoCarb will be in geostationary
orbit over the Americas and will provide subdaily observations from 45◦ S to 55◦ N.
Several narrow-swath instruments in Table 1 are selective in their observations to focus on specific targets and
avoid cloudy conditions. The GHGSat instruments observe
selected 12×12 km2
scenes with 25×25 m2 pixel resolution
and instrument pointing to increase the signal-to-noise ratio (SNR). Carbon Mapper will observe 18 km swaths with
imaging strips as long as 1000 km in push-broom mode
and shorter strips in target-track (instrument-pointing) mode.
GHGSat has six satellites in orbit as of this writing to achieve
frequent return times, and Carbon Mapper similarly plans a
constellation of satellites. WorldView-3 observes scenes of
dimensions up to 66.5×112 km2
. MethaneSAT will observe
200 × 200 km2
targets in oil and gas systems and agricultural regions with 130 × 400 m2 pixel resolution, enabling
high-resolution quantification of regional emissions as well
as imaging of large point sources.
All area flux mappers in Table 1 have fine (< 0.5 nm) spectral resolution to enable precise measurements of methane
concentrations, traded against coarser (0.1–10 km) spatial
resolution. GHGSat achieves a combination of fine spatial
resolution and fine spectral resolution by instrument pointing. Most other point source imagers in Table 1 are designed
to observe land surfaces, which requires fine spatial resolution (< 50 m) but less stringent spectral resolution. These
instruments have serendipitous capability to detect methane
plumes in the broad 2.3 µm band, including hyperspectral
sensors with ∼ 10 nm spectral resolution (PRISMA, EnMAP,
EMIT; Cusworth et al., 2019) and even multispectral sensors
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9620 D. J. Jacob et al.: Quantifying methane emissions using satellites Table 1. Current and planned SWIR satellite instruments for observing atmospheric methane
a
.
Instrument Organization
b
date
Launch
size
pixel
Nadir Coverage Return
(d)
time
c (mum)
band
Methane
d (nm)
resolution
Spectral
e
Precision
f Reference
Area flux mappers
g
GOSAT
h
NIES
MOE,
JAXA, 2009 10 km
diameter
i
global 3 1.65,
2.3
j
0.06 0.7 % Parker et al. (2020);
Noël et al. (2022)
TROPOMI ESA 2017
k 5.5 × 7 km2 global 1 2.3 0.25 0.8 %l Lorente et al. (2021a)
GOSAT-GW
NIES
MOE,
JAXA, 2023 1 × 1–
10×10 km2
∗
+
global
targets
3 1.65 0.06 0.6 % NIES (2021)
MethaneSAT EDF 2023 130×400 m2 200×200 km2
targets
3–4 1.65 0.3 0.1 %–
0.2 %n (2021)
Rohrschneider et al.
Sentinel-5 ESA 2024 7.5×7.5 km2 global 1 1.65,
2.3
0.25 0.8 % ESA (2020)
GeoCarb NASA 2025 6 × 3 km2
America
N and S
o
0.5 2.3 0.2 0.3 %–
0.6 %
Moore et al. (2018)
CO2M ESA 2025 2 × 2 km2 global 5 1.65 0.3 0.6 % Sierk et al. (2019)
MERLIN
DLR
CNES, 2027 0.1×50 km2
p
global
28 1.65 3 ×
10
−4q
1.5 % Ehret et
al. (2017)
Point source imagers
r
Landsat-8
s USGS 2013 30 × 30 m2 global 16 2.3 200 30 %–
90 %t
Ehret et al. (2022)
WorldView-3 Digital
Globe
2014 3.7 × 3.7 m2 66.5 ×
112 km2
targets
< 1 2.3 50 6 %–
19 %t (2022)
Sanchez-Garcia et al.
Sentinel-2 ESA 2015 20 × 20 m2 global 2–5 2.3 200 30 %–
90 %t
Varon et al. (2021)
GHGSat
u
Inc.
GHGSat, 2016 25 × 25 m2 12 × 12 km2
targets
1–7
v 1.65 0.3 1.5 %
w Jervis et al. (2021)
PRISMA
x ASI 2019 30 × 30 m2 30 × 30 km2
targets
4 2.3 10 3 %–
9 %
Guanter et al. (2021)
EnMAP
x DLR 2022 30 × 30 m2 30 × 30 km2
targets
4 2.3 10 3 %–
9 %
Cusworth et al. (2019)
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9621 Table 1. Continued. EMIT NASA 2022 60 × 60 m2 Dust-emitting regions
y
3 2.3 9 2 %–
6 %z
Cusworth et al. (2019)
Carbon
Mapperaa
Carbon
Mapper
and
Planet
2023 30 × 30,
30 × 60 m2
18 km
swathsab
1–7
v 2.3 6 1.2 %–
1.5 %
Duren (2021)
a This table lists shortwave infrared (SWIR) satellite instruments currently operating or scheduled for launch that have documented methane-observing capabilities and offer
publicly accessible data (some for purchase; see Table 2). Instruments not yet launched are in italics, and launch dates are estimates as of this writing. All instruments are in
low-elevation polar sun-synchronous orbits except for GeoCarb, which will be in geostationary orbit over the Americas, and EMIT, which will be in an inclined precessing orbit.
All instruments measure SWIR solar radiation backscattered from the Earth’s surface except for MERLIN, which is a lidar instrument. The Gaofen-5 series of Chinese satellites
has capabilities similar to PRISMA and EnMAP (Irakulis-Loitxate et al., 2021) but is not included in the table because of the opacity of data acquisition and distribution.
A more comprehensive list of instruments, including from private companies with proprietary data, is available from GEO, ClimateTRACE, WGIC (2021).
b Organization abbreviations are as follows: JAXA is the Japan Aerospace Exploration Agency, MOE is the Ministry of Environment, NIES is the National Institute for Environmental Studies,
ESA is the European Space Agency, EDF is the Environmental Defense Fund, NASA is the National Aeronautics and Space Administration, CNES is the Centre National d’Etudes Spatiales,
DLR is the Deutsches Zentrum für Luft- und Raumfahrt, USGS is the United States Geological Survey, and ASI is the Agenzia Spaziale Italiana.
c Time interval between successive viewings of the same scene.
d Most useful band(s) for methane retrieval. The 1.65 and 2.3 µ
m bands have exploitable features at 1.63–1.70 and 2.2–2.4 µ
m, respectively.
e Full width at half maximum.
f Precision is reported as a percentage of the retrieved dry column methane mixing ratio
XCH4
.
g Area flux mappers are primarily designed to quantify total methane emissions on regional to global scales.
h The TANSO-FTS instrument aboard the GOSAT satellite. The instrument is commonly referred to as GOSAT in the literature. GOSAT-2 was launched in 2018 with
specifications similar to GOSAT but adding a 2.3 µ
m band (Suto et al., 2021).
i Circular pixels separated by about 270 km along-track and cross-track distance.
j The 2.3 µ
m band was added in GOSAT-2.
k TROPOMI was launched in October 2017, but the methane data stream begins in May 2018.
l The TROPOMI product reports a much higher precision of
∼ 2 ppb, but this only includes error from the measured radiances. Accounting for retrieval errors by validation with
TCCON data indicates a precision of 0.8 % (Schneising et al., 2019).
∗ Narrow-swath mode (1 × 1 to 3 × 3 km3 pixels) for urban regions and wide-swath mode (10 × 10 km2
) for global coverage.
n For 1–5 km binned data.
o From 45
◦ S to 55
◦ N.
p Integrating the signal along 50 km of the lidar orbit track.
q Lidar online and offline sampling at 1645.552 and 1645.846 nm, respectively.
r Point source imagers quantify emissions from individual point sources by imaging of the atmospheric plume.
s Landsat-9 was launched in 2021 with a similar capability to Landsat-8.
t For favorable (bright and spectrally homogeneous) surfaces.
u Including GHGSat-D (2016), GHGSat-C1 (2020), GHGSat-C2 (2021), and GHGSat-C3–GHGSat-C5 (2022). Plans are for six more launches in 2023.
v For the constellation. Individual satellites have return times of about 14 d.
w For the GHGSat-C satellites. GHGSat-D has a precision of 12 %–25 %.
x Other planned hyperspectral imaging spectrometers with observing capabilities similar to PRISMA and EnMAP include SBG and CHIME (Cusworth et al., 2019).
y EMIT is a surface mineral dust mapper that will fly on the International Space Station in a 51.6
◦ inclined orbit and will target arid areas.
z Based on the precision of PRISMA (Guanter et al., 2021) and the higher spectral resolution of EMIT (Cusworth et al., 2019).
aa Carbon Mapper is expected to be a constellation of satellites with two launches in 2023 and a goal of six launches in 2024.
ab Carbon Mapper push-broom mode has imaging strips as long as 1000 km with 30 × 60
m2 pixels. Carbon Mapper target-tracking mode has shorter imaging strips with 30 × 30
m2 pixels and
ground-motion compensation to achieve higher signal-to-noise ratio (lower detection threshold).
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9622 D. J. Jacob et al.: Quantifying methane emissions using satellites
Table 2. Attributes and data availability for satellite instruments observing atmospheric methanea
.
Instrument Attributes Data availabilityb
Area flux mappers
GOSAT Long-term record of high-quality data L2, open
TROPOMI Global continuous daily coverage L2, open
GOSAT-GW High-resolution mapping of urban areas L2, open
MethaneSAT High-resolution mapping of oil/gas/agricultural
source regions with imaging of large point
sources
L1, L2, and L4, openc
Sentinel-5 Global continuous daily coverage including the
1.65 µm band
L2, open
GeoCarb Continuous coverage for methane-CO2-CO
over North and South America with subdaily
observations
L2, open
CO2M High-resolution global continuous coverage L2, open
MERLIN Arctic and nighttime observations L2
Point source imagers
Sentinel-2, Landsat Global continuous data acquisition,
long-term records
L1, open
WorldView-3 Very high spatial resolution L1, for purchase
GHGSat High sensitivity (∼ 100 kg h−1
),
established constellation
L2 and L4, for purchased
PRISMA, EnMAP Medium sensitivity (100–1000 kg h−1
),
extensive coverage
L1, free on request
EMIT Medium sensitivity (100–1000 kg h−1
),
extensive coverage of low-latitude arid regions
L1, opene
Carbon Mapper High sensitivity (∼ 100 kg h−1
),
high observing system completeness
L2 and L4, openf
a See Table 1 for the specifications of each instrument. Instruments not yet launched are in italics.
b L1 (Level 1) indicates measured radiances, L2 indicates retrieved column dry mixing ratio XCH4
, L4 indicates derived emission
rates.
c L1 and L2 data will be made available upon request.
d Data may also be obtained from space agencies through agreements negotiated with GHGSat.
e Generation of an L2 product is under discussion.
f L1 data will be available for purchase.
with a single 2.3 µm channel (Sentinel-2, Landsat; Varon et
al., 2021) or a few channels (WorldView-3; Sanchez-Garcia
et al., 2022). Carbon Mapper will have 6 nm spectral resolution, which increases precision appreciably relative to 10 nm
(Cusworth et al., 2019).
All area flux mappers in Table 1 have an open data policy allowing free access from a distribution website or from
the cloud. The data are generally provided as XCH4
retrievals
(Level 2 or L2). MethaneSAT will distribute its data publicly
as inferred methane fluxes (L4), with the L1 and L2 data also
available upon request. Data access for point source imagers
is presently less straightforward. Sentinel-2 and Landsat have
freely accessible channel radiance (L1) data, but users must
perform their own methane retrievals and source rate estimates. GHGSat and WorldView-3 make observations at the
request of paying customers, with GHGSat providing column density (L2) and source rate (L4) data and WorldView-3
providing L1 data. PRISMA and EnMAP make observations
upon request from the scientific community and stakeholders, and the resulting L1 data are then freely accessible, but
again users must perform their own methane retrievals. Carbon Mapper will provide open L2 and L4 data.
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9623
Figure 1. Satellite instruments for observation of methane in the shortwave infrared (SWIR). Area flux mappers are designed to quantify
total methane emissions on regional to global scales. Point source imagers are designed to quantify emissions from individual point sources
by imaging the atmospheric plumes. Specifications for each instrument are in Tables 1 and 2. Satellite icons were obtained from https:
//www.gosat.nies.go.jp (last access: 22 July 2022) for GOSAT; Wikipedia Commons for TROPOMI, EMIT (International Space Station),
and Sentinel-2; https://space.skyrocket.de (last access: 22 July 2022) for GOSAT-GW, MERLIN, CO2M, and Carbon Mapper; https://www.
methanesat.org (last access: 22 July 2022) for MethaneSAT; ESA (2020) for Sentinel-5; https://www.ou.edu/geocarb/mission (last access:
22 July 2022) for GeoCarb; https://www.planetek.it/ (last access: 22 July 2022) for PRISMA; https://www.ghgsat.com/ (last access: 22 July
2022) for GHGSat; https://www.enmap.org/mission (last access: 22 July 2022) for EnMAP; https://directory.eoportal.org (last access: 22
July 2022) for WorldView-3; and https://www.usgs.gov/landsat-missions (last access: 22 July 2022) for Landsat.
2.2 Retrieval methods
The “full-physics” retrieval of methane columns from satellite SWIR spectra involves inversion of the spectra with a radiative transfer model (Butz et al., 2012; Thorpe et al., 2017).
It typically solves simultaneously for the vertical profile of
methane concentration, the vertical profile of aerosol extinction, and the surface reflectivity. Although the vertical profile
of methane may be retrieved in the inversion, there is actually no significant information on vertical gradients, and only
XCH4
is reported together with an averaging kernel vector
for sensitivity to the vertical profile (near unity in the troposphere). The retrieval may fail if the atmosphere is hazy
or if the surface is heterogeneous or too dark. Full-physics
TROPOMI retrievals in the 2.3 µm band thus have only a
3 % global success rate over land (Lorente et al., 2021a)
with large variability depending on location (Fig. 2). Arid
areas and midlatitudes are relatively well observed. Observations are much sparser in the wet tropics because of extensive cloudiness and dark surfaces and in the Arctic because
of seasonal darkness, extensive cloudiness, and low sun angles. Observations at high latitudes are very limited outside
of summer, resulting in a seasonal sampling bias.
The 1.65 µm band allows the alternative CO2 proxy retrieval, taking advantage of the adjacent CO2 absorption band
at 1.61 µm (Frankenberg et al., 2005). In this method, CH4
and CO2
are retrieved simultaneously without accounting
for atmospheric scattering, and XCH4
is then derived as
XCH4 =

CH4
CO2

XCO2
, (1)
where XCO2
is independently specified, typically from assimilated observations or from a global chemical transport model
(Parker et al., 2020; Palmer et al., 2021). The CO2 proxy
method takes advantage of the lower variability of CO2 than
methane and of the low CO2 co-emission from the dominant
methane sources (livestock, oil and gas systems, coal mining,
landfills, wastewater treatment, rice cultivation, wetlands). It
is much faster than the full-physics retrieval, achieves similar
precision and accuracy (Buchwitz et al., 2015), and largely
avoids biases associated with surface reflectivity and aerosols
because these biases tend to cancel in the CH4
/CO2
ratio.
It is subject to errors from unresolved variability of CO2 such
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9624 D. J. Jacob et al.: Quantifying methane emissions using satellites
Figure 2. Global TROPOMI observations of methane for December 2019–February 2020 and June–August 2020. Data are from the version
2.02 product, filtering out low-quality retrievals (qa_value < 0.5) and snow and ice surfaces diagnosed by blended albedo > 0.8 (Lorente
et al., 2021a). The top panels show the mean dry methane column mixing ratios XCH4
on a 0.1
◦ × 0.1
◦ grid. The middle panels show the
observation density as the number of successful observations per 1◦×1
◦ grid cell for the 3-month periods. The bottom panels show the mean
XCH4
differences between colocated TROPOMI and GOSAT observations plotted on a 2◦ × 2.5
◦ grid and adjusted upward by 10.5 ppb to
account for TROPOMI being 10.5 ppb lower than GOSAT in the global mean. GOSAT data are from the CO2 proxy retrieval version 9.0 of
Parker et al. (2020).
as in urban regions and is also subject to bias for sources
that co-emit methane and CO2 such as flaring and other incomplete combustion. The GOSAT instrument operating at
1.65 µm with 10 km pixels has a 24 % success rate over land
using the CO2 proxy retrieval, mainly limited by cloud cover
(Parker et al., 2020).
A limitation in using the 1.65 µm band is that it is narrower, with fewer spectral features and weaker absorption
than the 2.3 µm band, and it therefore requires an instrument with sub-nanometer spectral resolution (Cusworth et
al., 2019; Jongaramrungruang et al., 2021). The 2.3 µm band
can be successfully sampled for a full-physics retrieval by
hyperspectral instruments with ∼ 10 nm spectral resolution
(Thorpe et al., 2014, 2017; Cusworth et al., 2021a; Borchardt et al., 2021; Irakulis-Loitxate et al., 2021). Precision
improves with spectral resolution (Cusworth et al., 2019;
Jongaramrungruang et al., 2021) and with spectral positioning relative to the methane absorption lines (Scaffuto et al.,
2021). Multispectral instruments with one or several broadband channels (∼ 100 nm bandwidth) do not allow a spectrally resolved retrieval, but a simple Beer’s law retrieval
of the methane column enhancement in a plume relative to
background can still be achieved in the 2.3 µm band by inferring surface reflectivity from adjacent bands or from views of
the same scene when the plume is absent (Varon et al., 2021;
Sanchez-Garcia et al., 2022).
Yet another approach for retrieving methane enhancements from point sources is the matched-filter method in
which the observed spectrum is fitted to a background spectrum convolved with a target methane absorption spectrum
capturing the 2.3 µm absorption band (Thompson et al.,
2015; Foote et al., 2020). Matched filter methods have been
extensively used for mapping methane point sources from
airborne hyperspectral campaigns (Frankenberg et al., 2016;
Duren et al., 2019; Cusworth et al., 2021b) and have also
been used for satellite retrieval of point sources (Thompson et al., 2016; Guanter et al., 2021; Irakulis-Loitxate et al.,
2021). These methods directly retrieve the methane enhancement above background and are faster than a full-physics
retrieval. They are well-suited methods for plume imaging,
where the methane enhancement above local background is
the quantity of interest.
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9625
2.3 Precision and accuracy
Retrievals of XCH4 may be affected by random error (precision) and systematic error (bias or accuracy). A uniform
bias is inconsequential because it can be simply subtracted.
Random error is reducible by temporal averaging if the observation density is high. The most pernicious error is spatially variable bias, often called relative bias (Buchwitz et
al., 2015), which is generally caused by aliasing of surface
reflectivity spectral features into the methane retrieval. Variable bias corrupts the retrieved concentration gradients and
produces artifact features that may be wrongly attributed to
methane.
Area flux mapper instruments are generally validated
by reference to the highly accurate XCH4 measurements
from the worldwide Total Carbon Column Observing Network (TCCON) of ground-based sun-staring spectrometers
(Wunch et al., 2011). Variable bias can be estimated as the
spatial standard deviation across TCCON sites of the temporal mean bias (Buchwitz et al., 2015). Schneising et al. (2019)
inferred in this manner a global bias of −1.3 ppb for the
TROPOMI University of Bremen methane retrieval, a precision of 14 ppb, and a variable bias of 4.3 ppb. Lorente et
al. (2021a) inferred a global mean bias of −3.4 ppb and a
variable bias of 5.6 ppb for the current TROPOMI version
2 Netherlands Institute for Space Research (SRON) operational retrieval. Figure 3 places these values in the context of
TROPOMI observations over the Permian Basin oil field in
Texas and New Mexico. A typical single day of TROPOMI
observations shows large areas of missing and noisy data,
and thus temporal averaging is necessary, which also reduces
the random error. Averaging TROPOMI observations over
a month shows full coverage of the Permian with enhancements of ∼ 50 ppb over the principal areas of oil and gas
production, well above the variable bias of the instrument.
Reliance on the TCCON network to diagnose variable bias
is limited by the sparsity of network sites, almost all of which
are at northern midlatitudes. An alternative way is by reference to GOSAT. The current version 9 GOSAT retrieval
using the CO2 proxy method has a variable bias of only
2.9 ppb referenced to TCCON and is recognized as a wellcalibrated measurement (Parker et al., 2020). Spatial variability in the mean TROPOMI–GOSAT difference provides
a global assessment of TROPOMI variable bias (Qu et al.,
2021). Results in Fig. 2 (bottom panel), after correcting for
a global mean TROPOMI–GOSAT difference of −10.5 ppb
(TROPOMI lower than GOSAT), show that TROPOMI variable biases can exceed 20 ppb in some regions. The reason for such large biases relative to GOSAT is TROPOMI’s
coarser spectral sampling of the SWIR region, as well as the
unavailability of the CO2 proxy retrieval at 2.3 µm. Comparing TROPOMI and GOSAT observations for a region of interest is good practice before interpreting TROPOMI data for
that region (Z. Chen et al., 2022).
Variable bias is also a concern for point source imagers,
where it manifests as artifact features that could be mistaken
for methane plumes (Ayasse et al., 2018). This is of particular concern for heterogeneous surfaces (Cusworth et al.,
2019). Artifacts can be screened by visual inspection of the
candidate plumes in relation to wind direction, known infrastructure, and surface reflectivity (Guanter et al., 2021).
Machine-learning methods can also be trained to detect
plumes and recognize artifact noise patterns (Jongaramrungruang et al., 2022). Figure 3 shows illustrative observations
of point sources from Sentinel-2, PRISMA, and GHGSat in
the Permian Basin. The observations have lower precision
than TROPOMI (Table 1), but the methane enhancements are
much larger because the pixels are smaller. Point source detection thresholds and their relationship to precision are discussed in Sect. 5.
3 Global, regional, and point source observations
Figure 4 classifies the satellite instruments of Table 1 in terms
of their abilities to observe methane on global and regional
scales as area sources (area flux mappers) or on the scale
of individual point sources (point source imagers). Observations of these different scales target complementary needs
for our understanding of methane, and they correspondingly
have different observing requirements. Area sources may
integrate a very large number of individually small emitters that cumulate to a large total, such as low-production
oil wells (Omara et al., 2022). A practical definition of a
methane point source for our purposes, following Duren et
al. (2019), is a single facility emitting more than 10 kg h−1
over an area less than 30 × 30 m2
. This represents a typical limit of detection from aircraft remote sensing combined with a typical spatial resolution for point source imagers. With this definition of source threshold, Cusworth et
al. (2022) find on average that 40 % of emissions from US oil
and gas fields originate from point sources. This emphasizes
the need for characterizing methane emissions complementarily both as area sources and as point sources.
3.1 Global and regional observations with area flux
mappers
Global observation of methane targets the central question
of why atmospheric methane has almost tripled since preindustrial times and why it continues to increase. Ground
network measurements such as from NOAA are the reference for observing global trends because of their high accuracy (Bruhwiler et al., 2021), and some sites include isotopic
or other information to separate contributions from different
source sectors (Lan et al., 2021). But satellites have an essential role to play because of their dense and global coverage. They can identify the regions that drive the global trend
(Zhang et al., 2021). They have a unique capability to evaluate the accuracy and trends of methane emissions reported
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9626 D. J. Jacob et al.: Quantifying methane emissions using satellites
Figure 3. Satellite observations of atmospheric methane over the Permian Basin (Texas and New Mexico) in July 2020. The left panel shows
typical TROPOMI observations for 1 d (July 15), featuring large areas of missing data where the retrieval was not successful because of
cloud cover or other factors. The middle panel shows monthly mean TROPOMI observations on a 0.1
◦ × 0.1
◦ grid, featuring enhancements
over the Delaware and Midland basins where oil production is concentrated. TROPOMI data are from the version 2.02 retrieval of Lorente
et al. (2021a). The right panel shows sample observations of plumes from point sources by Sentinel-2, PRISMA, and GHGSat superimposed
on surface imagery from © Google Earth. Plume dimensions and inferred point source rates (Q) are given as an inset. See Sect. 4.2 for the
inference of point source rates from plume observations.
Figure 4. Classification of satellite instruments by their capability to observe atmospheric methane on global scales, on regional scales with
high resolution, and for point sources. Specifications for the satellite instruments are listed in Table 1, and key attributes are listed in Table 2.
Point source detection thresholds are given here as orders of magnitude. These detection thresholds are discussed in Sect. 5.2. Instruments
not yet launched are in italics.
by individual countries to the UNFCCC (Janardanan et al.,
2020) and thus contribute to the transparency framework of
the Paris Agreement (Deng et al., 2022; Worden et al., 2022).
Global observation of methane from space is presently
available from GOSAT and TROPOMI. GOSAT provides a
continuous and well-calibrated record going back to 2009
(Parker et al. 2020). Inversions of GOSAT data have been
used to attribute the contributions of different source regions
and sectors to the methane increase over the past decade
(Maasakkers et al., 2019; Chandra et al., 2021; Palmer et al.,
2021; Zhang et al., 2021). The TROPOMI data stream begins
in May 2018 and is much denser than GOSAT, but the ability
to use TROPOMI data in global inversions is presently limited by large variable biases in some regions of the world (Qu
et al., 2021; Fig. 2). This is likely to improve with future retrieval versions and may be overcome with careful data selection. Continuity of global methane observations from space
is expected over the next decade with the GOSAT series
(GOSAT-2, GOSAT-GW), Sentinel-5, and CO2M (Table 1).
MERLIN could make an important contribution toward betAtmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9627
ter understanding of methane emissions in the Arctic, which
is otherwise difficult to observe from space.
There is considerable interest in using satellite observations to quantify methane emissions with high resolution on
regional scales. This is important for reporting of emissions
at the national or sub-national state level, for monitoring oil
and gas production basins, and for separating contributions
from different source sectors. Oil and gas production basins
are typically a few hundred kilometers in size and may contain thousands of point sources that are individually small
but add up to large totals and are best quantified on a regional scale (Lyon et al., 2015). Several field campaigns using surface and aircraft measurements have targeted oil and
gas fields in North America (Karion et al., 2015; Pétron et al.,
2020; Lyon et al., 2021), but these campaigns are necessarily
short and are not practical in many parts of the world.
TROPOMI with its 5.5×7 km2 pixel resolution and global
continuous daily coverage is presently the only satellite instrument capable of high-resolution regional mapping of
methane emissions. GOSAT data are too sparse. TROPOMI
has been used to quantify emissions from oil and gas production fields including the Permian Basin (Zhang et al., 2020),
other fields in the US and Canada (Shen et al., 2022), and the
Mexican Sureste Basin (Shen et al., 2021), revealing large
underestimates in the bottom-up inventories. It has also been
used to quantify total methane emissions from China and to
attribute them to source sectors (Z. Chen et al., 2022). The
variable bias problems that affect global TROPOMI inversions can be less problematic on the scale of source regions
where methane enhancements are large, the bias may be less
severe (Fig. 2), and bias correction is possible through adjustment of boundary conditions in the transport model (Shen
et al., 2021). Capability for regional mapping of methane
emissions is expected to greatly expand in the future with
the MethaneSAT, GOSAT-GW, Sentinel-5, and CO2M instruments.
3.2 Point source observations with point source imagers
Monitoring large point sources is important for reporting of
emissions, and detection of unexpectedly large point sources
(super-emitters) can enable prompt corrective action. In situ
sampling and remote sensing from aircraft has been used extensively to quantify point sources (Frankenberg et al., 2016;
Lyon et al., 2016; Duren et al., 2019; Hajny et al., 2019;
Y. Chen et al., 2022; Cusworth et al., 2022) but is limited
in spatial and temporal coverage. Satellites again have an
essential role to play. They have enabled the discovery of
previously unknown releases (Varon et al., 2019; Lauvaux
et al., 2022) and the quantification of time-integrated total
emissions from gas well blowouts (Cusworth et al., 2021a;
Maasakkers et al., 2022a).
Observing point sources from space has unique requirements. Plumes are typically less than 1 km in size (Frankenberg et al., 2016; Fig. 3), thus requiring satellite pixels finer
than 60 m (Ayasse et al., 2019). It is desirable to quantify
emissions from single overpasses, though temporal averaging of plumes to improve SNR is possible with wind rotation if the precise location of the source is known (Varon et
al., 2020). The emissions are temporally variable, motivating frequent revisit times that can be achieved by a constellation of instruments. On the other hand, precision requirements are less stringent than for regional or global observations because of the larger magnitude of the concentration
enhancements.
The potential for space-based land imaging spectrometers
to detect methane point sources was first demonstrated with
the hyperspectral Hyperion instrument for the Aliso Canyon
blowout (Thompson et al., 2016). Hyperspectral sensors such
as PRISMA and others of similar design have since proven
capable of quantifying point sources of ∼ 500 kg h−1
(Cusworth et al., 2021a; Guanter et al., 2021; Irakulis-Loitxate
et al., 2021; Nesme et al., 2021). The first satellite instrument dedicated to quantifying methane point sources was the
GHGSat-D demonstration instrument launched in 2016 with
50×50 m2
effective pixel resolution and a precision of 12 %–
25 % depending on surface type (Jervis et al., 2021). Varon
et al. (2019) demonstrated the capability of that instrument
for discovering and quantifying persistent point sources in
the range 4000–40 000 kg h−1
in an oil and gas field in Turkmenistan. Five follow-up GHGSat instruments with precisions of 1 %–2 % were subsequently launched in 2020–2022,
building up to a constellation with frequent return times.
Multispectral instruments such as Sentinel-2, Landsat, and
WorldView-3 are also capable of detecting and quantifying
very large point sources (Varon et al., 2021; Ehret et al.,
2022; Sanchez-Garcia et al., 2022; Irakulis-Loitxate et al.,
2022a). Sentinel-2 and Landsat provide global and freely
accessible data that could form the foundation of a global
detection system for super-emitters (Ehret et al., 2022). A
large-scale survey of point emissions across the western
coast of Turkmenistan was achieved with the combination
of Sentinel-2 and Landsat (Irakulis-Loitxate et al., 2022a).
Detection of methane plumes from space has mainly been
over bright land surfaces. Observation of offshore plumes
such as from oil and gas extraction platforms is more difficult because of the low reflectance of water in the SWIR.
The signal can be enhanced by observing in the sunglint
mode, in which the sensor captures the solar radiation specularly reflected by the water. The sunglint observation configuration can be achieved by agile platforms able to point
in the Sun-surface forward scattering direction (PRISMA,
Worldview-3, GHGSat, Carbon Mapper) or by instruments
with a field-of-view sufficiently large that part of the swath
falls in the forward scattering area (TROPOMI, Sentinel-2,
Landsat). Irakulis-Loitxate et al. (2022b) demonstrated the
ability of sunglint retrievals from WorldView-3 and Landsat8 to detect large plumes from offshore platforms in the Gulf
of Mexico.
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9628 D. J. Jacob et al.: Quantifying methane emissions using satellites
The capability to monitor methane point sources from
space is expected to expand rapidly in coming years through
the GHGSat and Carbon Mapper constellations as well as
new hyperspectral missions (Cusworth et al., 2019). Expanding constellations observing with frequent return times and
at different times of day will enable better understanding of
the intermittency of methane emissions. In an aircraft survey of the Permian Basin, Cusworth et al. (2021b) found that
individual point sources produced detectable emissions only
26 % of the time on average. Similar intermittency was observed for oil and gas facilities in California (Duren et al.,
2019). Allen et al. (2017) and Vaughn et al. (2018) point
out that some emissions from the oil and gas infrastructure
are highly intermittent by design (liquids unloading, blowdowns, and startups) and may have predictable diurnal variations. Emissions due to equipment failure may be persistent
(leaks, unlit flares), sporadic (responding to gas pressure), or
single events (accidents). An increased frequency of observation can identify persistence of emissions to enable corrective action, and better understanding of point sources that
are intermittent by design can lead to better quantification
of time-averaged emissions. Beyond this short-term intermittency, there is also long-term variability related to operating
practices and facility life cycle (Cardoso-Saldaña and Allen,
2020; Johnson and Heltzel, 2021; Varon et al., 2021; Allen
et al., 2022; Ehret et al., 2022), stressing the importance of
sustained long-term monitoring.
4 Inferring methane emissions from satellite
observations
Inferring methane emissions from satellite observations of
methane columns involves different methods for area flux
mappers and point source imagers. Area flux mappers are
typically used to optimize 2-D distributions of emissions on
regional or global scales by inverse methods. Point source
imagers are used to infer individual point source rates by
some form of mass balance analysis.
4.1 Global and regional inversions with area flux
mappers
Area flux mappers produce 2-D fields of methane observations from which to optimize 2-D fields of gridded emission
fluxes. The optimization involves an atmospheric transport
model (forward model) to relate emissions to the observed
concentrations. The optimal emissions are generally obtained
by Bayesian inference, fitting the observations to the forward
model and including prior estimates of emissions to regularize the solution where the observations provide insufficient
information (Brasseur and Jacob, 2017). Optimizing temporal trends of emissions can be done as part of the solution or
sequentially using a Kalman filter (Feng et al., 2017).
The basic procedure is as follows. Given an ensemble of
observations over a domain of interest assembled in an observation vector y, the task is to optimize the distribution of
emission fluxes assembled in a state vector x of dimension
n. The relationship between x and y can be assumed linear
for methane, despite the sensitivity of OH concentrations to
methane concentrations. This is because the inversion does
not significantly change the global methane concentration,
which is set by observation; furthermore, for regional inversions, the timescale for ventilation of the regional domain is
much shorter than that for chemical loss. Global inversions
often optimize OH concentrations as part of the state vector
and that relationship can also be assumed linear. Further assuming Gaussian error probability density functions (pdfs)
for x and y, the optimal (posterior) estimate of x is obtained
by minimizing a Bayesian cost function J (x) of the form
(Brasseur and Jacob, 2017):
J (x) = (x − xA)
T S
−1
A
(x − xA)
+ γ (y − Kx)
T S
−1
O
(y − Kx), (2)
Here xA is the prior estimate of emissions, SA is the corresponding prior error covariance matrix, K = ∂y/∂x is the
Jacobian matrix describing the sensitivity of observations to
emissions as given by the atmospheric transport model, SO
is the observational error covariance matrix including contributions from instrument and transport model errors, and γ
is a regularization parameter that may be needed to correct
overfit caused by imperfect definition of SO (Lu et al., 2021).
Since the relationship between x and y is linear, K fully defines the atmospheric transport model for the inversion. Jacob et al. (2016) describe alternative formulations for the
cost function such as in geostatistical inverse modeling where
prior information is provided as the relative spatial distribution of emissions rather than emission magnitudes (Miller et
al., 2020).
Specification of the error covariance matrices SA and SO
strongly affects the solution. Construction of SA can be done
by intercomparing bottom-up inventories (Maasakkers et al.,
2016; Bloom et al., 2017) or by using error estimates generated by the bottom-up inventories (Scarpelli et al., 2020).
Construction of SO can be done by the residual error method
in which the observations are compared to simulated concentrations from the atmospheric transport model with prior
emission estimates, and the residual difference after removing the mean bias is taken to be the observational error (Heald
et al., 2004; Wecht et al., 2014). The observational error for
satellites is generally found to be dominated by the instrument retrieval error rather than by the transport model error,
whereas for in situ observations it is dominated by the transport model error (Lu et al., 2021).
Minimization of the cost function J (x) in Eq. (2) to obtain the posterior solution xˆ and its error covariance matrix
Sˆ can be done either numerically or analytically (Brasseur
and Jacob, 2017). Sˆ and the related averaging kernel matrix
A = ∂xˆ /∂x = In − SSˆ −1
A
(Rodgers, 2000) determine the information content from the observations and the ability of
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9629
the inversion to improve on the prior estimate. The diagonal
terms of A ranging from 0 to 1 are called the averaging kernel sensitivities and measure the ability of the observations
to constrain the solution for that state vector element independently of the prior estimate (1 = fully, 0 = not at all). The
trace of A is called the degrees of freedom for signal (DOFS)
and represents the total number of pieces of information that
can be fully constrained from the observations. An inherent
assumption is that the observations, the transport model, and
the prior information are unbiased. Although the prior estimate is in principle unbiased since it represents our best estimate before the observations are taken, under-accounting of
SA together with incorrect spatial distribution of prior emissions can drive bias in inversion results (Yu et al., 2021).
Numerical solution for min(J (x)) using the adjoint of the
atmospheric transport model or other variational methods optimizes a state vector of any dimension by avoiding explicit
construction of the full Jacobian matrix K and may use various procedures to estimate Sˆ (Bousserez et al., 2015; Cho et
al., 2022). Analytical solution provides a closed-form expression for Sˆ but requires the computationally expensive construction of K column-by-column with n perturbation runs
of the atmospheric transport model. This limits the dimension and hence the resolution of the state vector that can be
optimized. However, once K has been constructed, inversion
ensembles can be conducted at no significant added computational cost to explore uncertainties in inversion parameters
or to examine the complementarity and consistency of different observation subsets such as from different satellite instruments or from ground-based sites (Lu et al., 2021, 2022).
This includes optimization of the regularization parameter
γ so that the sum of prior terms in the posterior cost function matches the expected value from the chi-square distribution, JA(xˆ) = (xˆ −xA)
T S
−1
A
(xˆ −xA) ∼ n (Lu et al., 2021). Increasing access to large computational clusters has facilitated
the construction of K as an embarrassingly parallel problem,
enabling analytical solution for state vectors with n > 1000
(Maasakkers et al., 2019). Nesser et al. (2021) show that even
larger dimensions can be accessed by approximating the Jacobian along leading patterns of information content.
Figure 5 illustrates the inversion of TROPOMI observations with a 1-month example for the Permian Basin using an analytical solution with 0.25◦ × 0.3125◦
(≈ 25 ×
25 km2
) resolution. This calculation was done on the Amazon Web Services (AWS) cloud with the Integrated Methane
Inversion (IMI) open-access facility for analytical inversions
of TROPOMI data, enabling users to directly access the
TROPOMI data archived on AWS and infer emissions for
their selected domain and time window of interest with precompiled inversion code (Varon et al., 2022).
The assumption of Gaussian error pdfs for prior emission estimates in Eq. (2) may not always be appropriate. A
log-normal distribution is often more correct (Yuan et al.,
2015) and can be accommodated in analytical inversions
(Maasakkers et al., 2019; Z. Chen et al., 2022). Brandt et
al. (2016) show that the log-normal distribution still underestimates the heavy tail of the frequency distribution of point
sources (the super-emitters). Application of inverse methods to detect and quantify individual super-emitters within
a source region (such as an oil and gas field) may require
a bimodal pdf for prior estimates, and an L1 norm cost
function may be better suited than the standard L2 norm of
Eq. (2) (Cusworth et al., 2018). A Markov chain Monte Carlo
(MCMC) method for the inversion as used by Western et
al. (2021) enables the specification of any prior and observational error pdfs and returns the full posterior error pdf on
emissions, but it is computationally expensive and its cost
increases rapidly as n increases.
The inversion typically optimizes a geographical 2-D array of emission fluxes, but quantifying emissions by source
sector is often of ultimate interest. Sectoral information is
generally contained in the prior inventory. The simplest approach is to assume that the posterior-to-prior emissions ratio for a given grid cell applies equally to all emissions in
that grid cell (Turner et al., 2015) or in a manner weighted
by the prior uncertainties of the different sectors (Shen et al.,
2021). The posterior error covariance matrix Sˆ and averaging
kernel matrix A on the 2-D grid can similarly be mapped to
specific sectors and/or be summed over a domain such as an
individual country (Maasakkers et al., 2019). A more general
approach for sectoral attribution introduced by Cusworth et
al. (2021c) maps the (xˆ, Sˆ) solution onto any alternative state
vector z (such as sector-resolved emissions) with its own
prior information (zA, ZA) to obtain a solution zˆ with posterior error covariance matrix Zˆ . This specifically allows for
comparisons of results from inversions using different prior
information.
4.2 Quantification of point sources with point source
imagers
Quantification of point sources from satellite observations
of instantaneous plumes poses a different kind of inversion
problem. In this case, a single quantity, the point source rate
Q [kg s−1
], is to be inferred from a single observation of
the plume. Figure 3 showed examples of plume observations.
The morphology of the instantaneous plume is determined by
turbulent diffusion superimposed on the mean wind, with a
plume boundary (commonly called the plume mask) defined
by the detection limit of the instrument. The observation is
of the total methane column and so is relatively insensitive
to vertical boundary layer mixing, which is a major source of
error in interpreting plumes from in situ aircraft observations
(Angevine et al., 2020). On the other hand, unlike for in situ
aircraft observations, there is no direct measurement of the
wind speed U in the plume. The lack of precise wind speed
information is a major source of error for interpreting satellite observations because concentrations in the plume vary as
the ratio Q/U, meaning that errors in U propagate proportionally to errors in Q.
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9630 D. J. Jacob et al.: Quantifying methane emissions using satellites
Figure 5. Integrated Methane Inversion (IMI) on the Amazon Web Services (AWS) cloud (Varon et al., 2022). The IMI accesses the
TROPOMI operational data posted on the cloud and carries out analytical inversions for user-selected domains and time periods. Before
conducting the inversion, users can run an IMI preview to visualize the observations, the default prior emission estimates (to which they can
substitute their own), the expected information content of the inversion (degrees of freedom for signal or DOFS), and the SWIR albedos
for indication of data artifacts. If the preview is satisfactory, they can then run the inversion to generate posterior emission estimates with
averaging kernel sensitivities indicating where the observations can successfully constrain emissions. Shown here is an example given by
Varon et al. (2022) for a 1-month (May 2018) inversion over the Permian Basin, using the prior emission estimate from the EDF inventory
(Zhang et al., 2020). The IMI is accessible at https://imi.seas.harvard.edu (last access: 23 July 2022).
Figure 6 summarizes different methods for inferring point
source rates from satellite observations of instantaneous
plumes. Details on these methods are given by Krings et
al. (2011), Varon et al. (2018), and Jongaramrungruang et
al. (2019, 2022). The Gaussian plume is the classic model
for turbulent diffusion from a point source, but it is valid only
for a plume sampling a representative ensemble of turbulent
eddies. Methane plumes are generally too small for this condition to be met (Jongarangmrungruang et al., 2019), as illustrated in Fig. 3 where the plume shapes are not Gaussian. A
simple mass balance method applying the local wind speed
to the methane enhancement observed in the plume is flawed
for sub-kilometer scales because ventilation is determined by
turbulent eddies more than by the mean wind (Varon et al.,
2018).
The Gauss theorem method, in which the source rate is
calculated as the outward flux summed along a contour surrounding the point source, is extensively used for in situ aircraft observations where concurrent measurements of wind
vector and methane concentration are available to calculate
the local flux as the aircraft circles around the source (Hainy
et al., 2019). In the absence of in situ wind data, one can apply a single estimate of the wind vector based on local station
or assimilated data (Krings et al., 2011). However, the calculation then does not account for the contribution of turbulent
diffusion to the outward flux. In addition, any sources within
the contour will alias into the inferred point source rate.
Two successful methods to derive point source rates from
observations of instantaneous plumes have been the crosssectional flux (CSF) method (White, 1976; Krings et al.,
2011), in which the source rate is inferred from the product
of the methane enhancement and the wind speed integrated
across the plume width, and the integrated mass enhancement (IME) method (Frankenberg et al., 2016; Varon et al.,
2018), in which the total mass enhancement in the plume
is related to the magnitude of emission with a parameterization dependent on wind speed. Both methods are widely
applied to the retrieval of point source rates from satellite
observations and they yield consistent results (Varon et al.,
2019). The CSF method is more physically based, and source
rates can be derived from cross sections at different distances
downwind to reduce error (Fig. 6). The contribution of turbulent diffusion to the flux can be neglected in the direction
of the wind following the slender plume approximation (Seinfeld and Pandis, 2016). However, the dependence on wind
direction is an additional source of error relative to the IME
method.
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9631
Figure 6. Seven different methods for inferring point source rates Q [kg s−1
] from satellite observations of instantaneous plumes of methane
column enhancements 1 [kg m−2
] relative to background. The methods involve (1) fit to a Gaussian plume, (2) local mass balance for
near-source pixels, (3) Gauss theorem with integration of the outward flux along a closed contour s, (4) cross-sectional flux (CSF) integral,
(5) integrated mass enhancement (IME) with independent wind speed information, (6) IME with wind speed inferred from the plume angular
width θ, and (7) machine learning applying a convolution neural network (CNN) to the plume image. Methods (1), (2), (4), and (5) are
described by Varon et al. (2018); method (3) is described by Krings et al. (2011); method (6) is described by Jongaramrungruang et al. (2019);
and method (7) is described by Jongaramrungruang et al. (2022). In the equations, x denotes the plume axis for transport by the mean wind
and y denotes the horizontal axis normal to the wind. The IME [kg] is the spatial integral of the methane column enhancement 1 over
the plume mask. The wind speed U is that relevant to transport of the plume, and in the IME method (4) it is parameterized as an effective
wind speed Ueff to include the effect of turbulent diffusion. The Gauss theorem and CSF methods require wind direction information. The
IME method (4) requires a characteristic plume size L that can be taken as the square root of the plume area (Varon et al., 2018) or the radial
plume length (Duren et al., 2019). The empirical dispersion parameter σy [m] in the Gaussian plume method (1) characterizes the spread of
the plume. n in the Gauss theorem method is the unit vector normal to the contour.
Both the CSF and IME methods require estimates of
wind speed relevant to plume transport. For the CSF method
this is the mean wind speed over the vertical depth of the
plume, which can be parameterized from the 10 m wind
speed (Varon et al., 2018) or interpolated from a database of
wind speed vertical profiles (Krings et al., 2011). The effective wind speed Ueff in the IME method accounts for the effect of turbulent diffusion in plume dissipation and can be parameterized as a function of an observable 10 m wind speed
by using large-eddy simulations (LES) of synthetic plumes
sampled with the instrument pixel resolution, plume mask
definition, and observing time of day (Varon et al., 2018).
The need for independent information on wind speed, either
from measurements at the point source location or from a
meteorological database, can dominate the error budget in
inferring source rates from the CSF and IME methods, and
typically limits the precision to 30 % (Varon et al., 2018). The
error is larger for weak winds, which tend to be more variable and smaller for strong steady winds. However, plumes
are less likely to be detectable in strong winds because of
dilution. Weak winds are thus favorable for plume detection
but can induce large error in source quantification.
Jongaramrungruang et al. (2019) showed that the morphology of an observed plume contains information on wind
speed, as long slender plumes are associated with high wind
speeds while short stubby plumes are associated with low
wind speeds. By using the plume angular width as a measure of wind speed, they were able to infer source rates
without independent wind information. Jongaramrungruang
et al. (2022) developed that idea further with a convolutional
neural network (CNN) approach trained on LES plume images to learn the source rate from the 2-D plume structure.
Application to synthetic plumes, as would be sampled by
the AVIRIS-NG aircraft instrument at 1–5 m pixel resolution,
showed a mean precision of 17 % and a detection threshold of 50 kg h−1 over spectrally homogeneous surfaces. This
method has not yet been applied to satellite observations
where coarser pixels would result in lower sensitivity and
where retrievals are more subject to artifacts.
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9632 D. J. Jacob et al.: Quantifying methane emissions using satellites
5 Detection thresholds
5.1 Area sources
Here we examine the ability of area flux mappers to detect
total methane emission fluxes from a target domain with a
desired spatial resolution. This can involve repeated observations of the domain over multiple passes to increase precision and observation density, as illustrated in Fig. 3. The
observation time required to detect a desired flux threshold
at a desired spatial resolution then depends on the instrument
precision, the spatial coverage, the fraction of successful retrievals, the pixel size, the variability of emissions, and the
return time.
Following the conceptual model of Jacob et al. (2016), the
methane column enhancement 1X [ppb] resulting from a
uniform emission flux E [kg km−2 h
−1
] over a square domain of dimension W [km] is given by
1X = αE W, (3)
with a scaling coefficient α = (Ma/MCH4
)g/p U where Ma
and MCH4
are the molecular weights of dry air and methane,
g is the acceleration of gravity, p is the surface pressure, and
U is the wind speed for ventilation of the domain. With the
units above and assuming p = 1000 hPa and U = 5 km h−1
,
we have α = 4.0 × 10−2
ppb km h kg−1
. An instrument with
pixel-level precision σI [ppb] can detect this emission flux
with a single measurement if 1X >> σI
, but this is often
not the case. Spatial and temporal averaging of observations
improves the effective precision, and this improvement goes
as the square root of the number of observations if the error
is random, uncorrelated, and representatively sampled (IID
conditions). The time required for detecting the mean emission flux E over a domain of dimension W with a signal-tonoise ratio of 2 is then given by
t = tRmax"
1,
1
F N
max
1,

2σ
1X2
!#, (4)
where tR is the return time of the instrument (time interval
between successive passes), N is the number of observations
within the domain per individual pass for instrument pixel
sizes D smaller than W (for continuous mapping and square
pixels, we have N = (W/D)
2
), F is the fraction of successful
retrievals, and σ [ppb] is the variability that results from both
the instrument precision and the spatial variability σX (D,W)
of the enhancement 1X sampled by the pixels within the
domain:
σ =
q
σ
2
I + σX(D,W)
2
. (5)
Equations (3)–(5) provide a simple conceptual framework
for evaluating the ability of area flux mappers to detect regional emissions of a certain magnitude and with a desired
spatial resolution. For illustration purposes, consider an objective to detect emissions at either 100 or 10 km resolution.
In the gridded version of the methane emission inventory
from the US Environmental Protection Agency (Maasakkers
et al., 2016), 75 % of total national anthropogenic emissions are contributed by 0.1
◦ × 0.1
◦
(≈ 10 × 10 km2
) grid
cells with emission flux E > 0.5 kg km−2 h
−1
, and 30 %
are contributed by grid cells with E > 5 kg km−2 h
−1
(Jacob et al., 2016). Shen et al. (2022) find a mean emission
of 0.18 Tg yr−1
for 12 major oil and gas production basins
in the US EPA inventory, which for a typical basin scale
of 200 × 200 km2
corresponds to a mean emission flux of
0.5 kg km−2 h
−1
. Taking E = 0.5 kg km−2 h
−1
as a desired
flux detection threshold on a 100 km scale, or alternatively
E = 5 kg km−2 h
−1
as a desired flux detection threshold on
a 10 km scale, we find from Eq. (3) a mean enhancement
1X = 2.0 ppb. Instrument precisions for the flux mappers in
Table 1 are in the range 3–15 ppb, and we assume that σX
is small in comparison. We further assume F = 0.24 for instruments operating at 1.65 µm by analogy with GOSAT using the CO2 proxy method (mainly limited by cloud cover)
and F = 0.03 for instruments operating at 2.3 µm by analogy with TROPOMI (limited by both cloud cover and spectrally inhomogeneous surfaces). Other instrument properties
are taken from Table 1.
Table 3 shows the results of this illustrative calculation. In
the 100 km resolution case we find that TROPOMI requires
a 4-week averaging period, limited by the small fraction
of successful retrievals. GOSAT-GW requires 18 d in global
viewing mode, as the greater fraction of successful retrievals
is offset by coarser pixels and 3 d return time. It requires only
one pass in target mode. MethaneSAT requires a single pass
and is limited by its 3 d return time. Sentinel-5 requires 5 d,
much shorter than TROPOMI despite coarser pixels because
it uses the 1.65 µm band. GeoCarb requires only 3 d because
of its twice-daily observations. CO2M requires only a single
pass and is limited by its 5 d return time. In the 10 km resolution case, we find that only MethaneSAT has an averaging
time of less than a week, with GOSAT-GW requiring 18 d in
target mode (limited by its lower instrument precision) and
other instruments requiring several months or more. However, both MethaneSAT and GOSAT-GW in target mode only
cover limited domains (200 × 200 km2
for MethaneSAT).
The above conceptual model is crude and overoptimistic,
assuming ideal reduction of errors and uncorrelated retrieval
success across instrument pixels, ignoring variable bias, and
taking instrument specifications from Table 1 at face value,
but it is useful for intercomparing instruments and it highlights critical variables determining detection thresholds for
different applications. The advantage of the 1.65 µm band is
readily apparent because it achieves a much higher success
rate through the CO2 proxy retrieval. The MethaneSAT instrument with high precision and small pixels is most useful
for quantifying fluxes at high spatial resolution. For coarser
resolutions, return time and spatial coverage can be more important considerations.
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9633
Table 3. Averaging time requirements for regional source detection by area flux mappersa
.
Instrument Averaging time Averaging time
E = 0.5 kg km−2 h
−1
, 100 × 100 km2 E = 5 kg km−2 h
−1
, 10 × 10 km2
TROPOMI 28 d > 1 year
GOSAT-GW 18 d (global), 3 d (target) 18 d (target)
MethaneSAT 3 d 5 d
Sentinel-5 5 d > 1 year
GeoCarb 3 d 1 year
CO2M 5 d 120 d
a
Illustrative calculation using the conceptual model of Eqs. (3)–(5) applied to the detection of an emission flux averaging
0.5 kg km−2 h
−1 over a desired spatial resolution of 100 × 100 km2
, or 5 kg km−2 h
−1 over a desired spatial resolution of
10 × 10 km2
. See the text for details and Table 1 for the specifications of the different instruments. Results for
GOSAT-GW are given for both global and target viewing modes. Instruments not yet launched are in italics.
5.2 Point sources
In the case of point source imagers, the detection threshold
applies to single-pass observations of the plumes. Table 4
lists point source detection thresholds reported in the literature for different instruments. Detection thresholds are defined by the ability to determine the plume mask against a
noisy background and to retrieve the corresponding emissions. The detection thresholds for a given instrument depend strongly on surface type and are lowest for bright,
spectrally homogeneous surfaces. They also depend on wind
speed, which complicates the definition of detection threshold because weak winds facilitate detection but cause large
error in quantification (Varon et al., 2018). The best range of
wind speeds to allow both detection and quantification is 2–
5 m s−1
(Varon et al., 2018). Sherwin et al. (2022) conducted
a series of controlled release experiments under those favorable surface and wind conditions and confirmed the ability
of GHGSat to quantify emissions down to 200 kg h−1
and
Sentinel-2, Landsat-8, PRISMA, and WorldView-3 to quantify emissions down to the 1400–4000 kg h−1
range.
For a given surface and wind speed, the main instrument
predictors of point source detection threshold are spatial resolution, spectral resolution, and precision. Finer spatial resolution decreases the dilution of the plume enhancements over
the pixel area, thus increasing the magnitude of the enhancements within plume pixels and facilitating detection. An
airborne imaging spectrometer observing from low altitude
such as AVIRIS-NG (with spatial resolution of 1–8 m depending on aircraft altitude) is thus much more sensitive than
satellite instruments with similar spectral resolution. Higher
spectral resolution increases precision and reduces the aliasing of surface spectral features into the methane retrieval
(Cusworth et al., 2019; Jongaramrungruang et al., 2021). For
hyperspectral and multispectral instruments, the spectral positioning of the bands relative to the methane absorption lines
is also important (Scaffuto et al., 2021; Sanchez-Garcia et
al., 2022). Precision depends on other instrument properties
beyond spectral resolution and positioning, including the capability of pointing to specific targets to increase the SNR
through longer sample collection. Pointing is how GHGSat
achieves a combination of high spatial and spectral resolution.
The detection thresholds in Table 4 are not strictly comparable between instruments because they reflect different levels of evidence. One may still usefully classify the
instruments by order-of-magnitude thresholds of ∼ 100,
∼ 500, and ∼ 1000–10 000 kg h−1
(Fig. 4). Instruments in
the ∼ 100 kg h−1
class include GHGSat, WorldView-3, and
Carbon Mapper. A typical point source imager with spatial resolution ∼ 30 m requires spectral resolution of 5 nm or
better to fit into this class (Cusworth et al., 2019), though
WorldView-3 can achieve this class for bright spectrally homogeneous surfaces through its combination of very high
spatial resolution (3.7 × 3.7 m2
) and favorable spectral positioning (Sanchez-Garcia et al., 2022).
Instruments in the ∼ 500 kg h−1
class include the land
hyperspectral sensors (PRISMA, EnMAP, EMIT) and
MethaneSAT. The land hyperspectral sensors have ∼ 30 m
spatial resolution and achieve this class with 10 nm spectral
resolution in the 2.3 µm band, enabling either a full-physics
or matched filter retrieval. MethaneSAT will have coarser
130 × 400 m2
spatial resolution but higher precision enabled
by 0.3 nm spectral resolution in the 1.65 µm band, with the
added benefit of allowing a CO2 proxy retrieval to minimize
artifacts.
Instruments in the 1000–10 000 kg h−1
class include the
multispectral land sensors Sentinel-2 and Landsat with
20–30 nm spatial resolution and a single measurement in
the 2.3 µm band to allow a simple Beer’s law retrieval.
TROPOMI can detect extremely large point sources or clusters of sources (> 25 000 kg h−1
) over its 5.5 × 7 km2 pixels
(Lauvaux et al., 2022), though coarse spatial resolution hinders source identification.
The relevance of measuring individual point sources at
these different thresholds can be assessed by considering
their contributions to total emissions. Cusworth et al. (2022)
find on average that 40 % of emissions from US oil and gas
fields originate from point sources > 10 kg h−1 detectable by
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9634 D. J. Jacob et al.: Quantifying methane emissions using satellites
Table 4. Point source detection thresholds for different satellite instrumentsa
.
Instrument Detection threshold (kg h−1
) Reference
TROPOMI 25 000b Lauvaux et al. (2022)
Sentinel-2, Landsat-8/9 1800–25 000c Varon et al. (2021); Ehret et al. (2022); Irakulis-Loitxate et al. (2022a)
PRISMA 500–2000d Guanter et al. (2021)
MethaneSAT 500 Christopher Chan Miller, Harvard University, personal communication, 2022.
GHGSat-D 1000–3000 Jervis et al. (2021)
GHGSat-C1, C2 100–200e Gauthier (2021)
Carbon Mapper 50–200f Duren (2021)
WorldView-3 < 100 Sanchez-Garcia et al. (2022)
AVIRIS-NG (aircraft)g 2–10h Duren et al. (2019)
a The detection thresholds are as reported in the references and are generally for favorable winds (< 5 m s−1
) and favorable surfaces (bright and spectrally homogeneous) unless
otherwise indicated. As pointed out in the text, weak winds are favorable for detection but not for quantification, and this places some ambiguity in the definition of detection
threshold. Specifications for each instrument are in Table 1. Instruments that are yet to be launched are in italics.
b From an ensemble of 1800 observed detections for TROPOMI 5.5 × 7 km2 pixels. The pixels may contain multiple point sources.
c Observations over surfaces ranging from bright and homogeneous (Sahara) to highly heterogeneous (farmland).
d From LES synthetic plumes and observations over surfaces ranging from Sahara (bright homogeneous surfaces) to Shanxi Province in China (darker more heterogeneous surfaces
with significant terrain).
e Verified by controlled releases (MacLean, 2021; Sherwin et al., 2022).
f 50 kg h−1
in target mode with pointing and 200 kg h−1
in push-broom mode.
g Airborne imaging spectrometer with spectral resolution of 5 nm and pixel resolution of 1–8 m depending on aircraft altitude (Thorpe et al., 2017).
h Observations in California with range determined by surface brightness.
AVIRIS-NG. Figure 7 shows the cumulative frequency distributions (CFDs) by number and total emission of point
sources larger than 10 kg h−1
sampled by airborne remote
sensing over California and over US oil and gas fields (Duren
et al., 2019; Cusworth et al., 2022). Results are shown for individual campaigns and for the combined CFD with equal
weighting between campaigns. A satellite instrument with
detection threshold of 100 kg h−1
could detect 50 %–95 % of
point sources depending on the region (80 % in the combined
data set), contributing 75 %–99 % of point source emissions
(95 % for the combined data set). An instrument with detection threshold of 1000 kg h−1
could detect 0 %–15 % of
point sources (5 % for the combined data set), contributing
0 %–55 % of point source emissions (30 % in the combined
data set). Brandt et al. (2016) find that sources in the 10–
100 kg h−1
range contribute 20 % of emissions from point
sources > 10 kg h−1
in their survey of emissions from US
oil and gas fields. The data set of Fig. 7 includes only a few
emitters in the ∼ 10 000 kg h−1
range. Global statistics of aircraft and satellite data suggest a power law frequency distribution of point source emissions with ∼ 100× fewer sources
at 10 000 kg h−1
than at 1000 kg h−1
(Ehret et al., 2022; Lauvaux et al., 2022). These so-called ultra-emitters could still
contribute significantly to total emissions in some regions.
6 Observing system completeness
Here we introduce the concept of observing system completeness as the capability of an instrument (or ensemble of
instruments) to fully quantify their target emissions within a
selected domain and time window. For area flux mappers the
target would be the total methane emissions within the domain at a desired spatial resolution, while for point source
imagers the target would be the total emissions within the
domain contributed by point sources larger than 10 kg h−1
.
6.1 Observing system completeness for area flux
mappers
Observations from area flux mappers are generally used to
infer 2-D distributions of total emissions over a regional domain of interest by Bayesian inference. The observing system completeness is then defined by the DOFS (Sect. 4.1
and Fig. 5). Given n state vector elements of emissions on
the 2-D grid, the DOFS tell us how many of those elements
are quantified by the observations, and the averaging kernel
sensitivities (diagonal terms of the averaging kernel matrix,
adding up to the DOFS) give that information for the individual state vector elements.
As pointed out by Nesser et al. (2021) and Varon et
al. (2022), it is possible to roughly estimate the DOFS of
an observing system for a selected domain and time period
without doing any actual forward model calculations. Consider a domain divided into n emission state vector elements
of individual dimension W [km], sampled with an instrument
providing m successful observations over the domain in the
selected time period. Let σA be the mean prior error standard
deviation for the individual state vector elements and σO the
mean observational error standard deviation. The DOFS can
then be estimated as
DOFS =
n σ2
A
σ
2
A +
(σO/k)
2
m
, (6)
where k = 1X/E [ppb km2 h kg−1
] is the Jacobian matrix element that relates the column mixing ratio enhanceAtmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9635
Figure 7. Cumulative frequency distributions (CFDs) of point source rates above 10 kg h−1
for 3879 point sources detected by airborne
remote sensing in California and in US oil and gas basins by Duren et al. (2019) and Cusworth et al. (2022). Many of the individual point
sources were detected multiple times, and the values entered in the frequency distributions are the averages of these detections not including
non-detection events; they thus represent the average emission from the source when on, as is relevant to the definition of the instrument
detection threshold CD in Eq. (8). The colored curves are for individual campaigns, and the black curve is the combined CFD for all regions
with equal weighting per campaign. The top panel gives the cumulative fraction of emissions contributed by detected point sources above
a given rate, and the bottom panel gives the cumulative fraction of the number of point sources. For example, a satellite instrument with
detection threshold of 100 kg h−1
could detect 80 % of the point sources in the combined CFD, contributing 95 % of total point source
emissions. An instrument with detection threshold of 1000 kg h−1
could detect 5 % of the point sources in the combined CFD, contributing
30 % of total point source emissions.
ment 1X [ppb] over a state vector element to the emission
flux E [kg km−2 h
−1
] for that element. Following Nesser et
al. (2021), we can approximate k with a simple mass balance
model as
k = η
Ma
MCH4
W g
U p
, (7)
where η is a coefficient to account for turbulent diffusion.
Nesser et al. (2021) and Varon et al. (2022a) find that η = 0.4
is a suitable value for W in the range 25–100 km. Further
assuming U = 5 km h−1
and p = 1000 hPa, we obtain k =
1.4×1010W [ppb km2 h kg−1
]. The mean prior error standard
deviation can be estimated as σA = f QA/(nW2
), where QA
is the total prior estimate of emission in the domain [kg h−1
]
and f is the fractional error (such as 50 %). For the example
of Fig. 5 with a 1-month inversion of TROPOMI observations over the Permian Basin, Varon et al. (2022) find that this
rough estimate prior to doing the inversions yields a DOFS of
11.7, close to the value of 10.8 found in the actual inversion.
The simple estimate of DOFS in Eq. (6) yields basic insights into the factors affecting observing system completeness for an area flux mapper. Instrument precision and number of observations (or observation density for a given area)
are critical. The bar for the observations to improve on the
prior estimate depends on the estimated error for that prior
estimate (smaller prior error means a higher bar for the observations). Increasing the requirement on spatial resolution
(large n, small W) leads to smaller absolute prior errors for
individual state vector elements and in turn raises the requirement on the precision and number of observations.
6.2 Observing system completeness for point source
imagers
Observing system completeness for a point source imager (or
a constellation) can be defined as its ability to quantify total emissions from point sources larger than 10 kg h−1 over
a selected domain and time window. Such completeness in
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9636 D. J. Jacob et al.: Quantifying methane emissions using satellites
observation of point sources is important not only for complementing the information from area flux mappers but also
for leak detection and repair (LDAR) programs where regularly surveying point sources in a region can enable prompt
action to fix malfunctioning equipment (Kemp et al., 2016;
Fox et al., 2021). Current LDAR programs rely on a combination of ground surveys, drones, and aircraft, but we will
see that satellites have an important role to play.
Let C ∈ [0,1] denote the observing system completeness
for point sources as the fraction of total point source emissions larger than 10 kg h−1 within a domain and time window
that can be detected by a given instrument (or constellation of
instruments). C is limited by a combination of the instrument
detection threshold (CD), spatial coverage (CS), and temporal sampling (CT):
C = CD × CS × CT. (8)
Here CD is the fraction of point source emissions that can be
detected on the basis of the instrument’s detection threshold,
as inferred for example from Fig. 7. CS is the fraction of the
domain that the instrument observes at least once within the
time window. If there is full spatial coverage within the time
window, CS = 1. CT = 1 − (1 − F p)
N is the probability for
an observed source to be actually detected within the time
window given the number N ≥ 1 of observations in the window, the source persistence p (fraction of time that the source
is emitting above the detection threshold), and the fraction F
of successful retrievals, taken here as the fraction of clearsky observations. For example, an intermittent source with
p = 0.2 that is observed with a 1-week return time and 30 %
clear skies would have CT = 0.96 for 1 year of observations
but CT = 0.23 for 1 month. If spatial coverage and observing frequency are sufficient, C is limited by the instrument’s
detection threshold (CD). If they are not (and depending on
source persistence and cloud cover), CS and CT may limit
observation system completeness rather than CD.
Figure 8 shows the frequency distribution of persistence
(p) for 2500 oil and gas point sources detected and quantified by the airborne AVIRIS-NG and Global Airborne Observatory instruments in US field campaigns (Cusworth et
al., 2022). The left panel shows the frequency distribution of
mean emissions from individual point sources for each persistence bin. From there we can estimate the observing system completeness for any instrument on the basis of its detection threshold, spatial coverage, and return time. The right
panel plots the resulting cumulative observing system completeness for the ensemble of 2500 point sources as achieved
by either (1) an airborne instrument with 10 kg h−1 detection threshold and bi-monthly (60 d) sampling interval or
(2) a satellite instrument with 100 kg h−1 detection threshold and bi-weekly (14 d) sampling interval. The calculation
is done for a 1-year time window with 30 % clear skies, assuming CS = 0.95 in both cases, and the cumulative results
are shown across the range of persistence bins. We see in
this example that the two observing systems have comparable success for persistent sources (p > 0.5) by trading CD for
CT, but the satellite system is better for intermittent sources
(p < 0.5), despite its higher detection threshold, because of
the greater benefit from frequent observations.
Figure 9 further illustrates the trade space between detection threshold and return time for determining observing system completeness. Results are for the ensemble of
2500 point sources with statistics given in Fig. 8. We see
from Fig. 9 that an observing system completeness of 0.6
can be achieved by an instrument with a detection threshold of 300 kg h−1
sampling weekly. Such an instrument performs as well as one with low detection threshold but sampling only every 2 months. Achieving an observing system
completeness higher than 0.8 requires an instrument with detection threshold better than 150 kg h−1
that samples at least
biweekly.
Our calculation of CT as presented above assumes that a
point source follows a binary emission frequency distribution
(on/off) with constant emissions when on. Actual sources
have more complex variability (Allen et al., 2022; Zimmerle
et al., 2022). Similar to the analysis of Sect. 5.1, a simple
analysis can be done by assuming Gaussian statistics following Hill and Nassar (2019) to estimate the number N of observations needed to quantify a mean point source emission
rate (1±δ)Q with relative precision of δ defined by the 95 %
relative confidence interval:
N =
1
F p
(1.96
σ
δ
)
2
, (9)
σ =
q
σ
2
I + σ
2
S
. (10)
Here σ is the standard deviation of individual measurements
determined by instrument precision (σI) and variability in the
source (σS). Using statistics from airborne surveys in the Permian Basin, we find that 71 observations per year (roughly
5 d return time, assuming 30 % clear skies) would be required
to estimate annual point source emissions from that highly
intermittent population within 50 % (p = 0.24, σI = 36 %,
σS = 45 %; Cusworth et al., 2021b). Increasing the required
annual emission precision to 35 % would require 145 observations per year (2 d return time). For a less intermittent population (p = 0.5), we find N = 43 (8 d return time) to achieve
50 % precision and N = 87 (4 d return time) to achieve 35 %
precision. These observing frequencies can be achieved with
a satellite constellation but would be challenging for an airborne program.
The tails of the pdfs for point source emissions are a particular challenge to sample representatively. The pdfs are generally heavy-tailed, resulting in a low estimate of mean emissions (Zimmerle et al., 2022), which may be addressed with
very dense sampling (Y. Chen et al., 2022) or with supporting
observations from area flux mappers. Persistence is defined
in the observations by the frequency of occurrence of emissions above the detection threshold, but non-detection could
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9637
Figure 8. Point source rates, persistence, and observing system completeness for an ensemble of 2500 oil and gas point sources sampled by
aircraft remote sensing in five US oil and gas basins (Cusworth et al., 2022). The left panel shows the frequency distribution of mean point
sources rates for different persistence bins (p, fraction of the time that the source is detected), where the mean is computed by assuming
zero emission when no plume is detected. Boxes and whiskers indicate 10th, 25th, 50th, 75th, and 90th percentiles. The right panel shows
the percentage of total point source emissions contributed by different persistence bins. Also shown in that panel is the cumulative observing
system completeness C = CD × CS × CT (Eq. 8) for 1 year of observations under 30 % clear-sky conditions and two observing systems,
one with 100 kg h−1 detection threshold and bi-weekly sampling (green line) and one with 10 kg h−1
and bi-monthly sampling (red line).
We assume spatial coverage CS = 0.95 for both. The observing system completeness is computed individually for each basin and then
averaged. Both observing systems have comparable performance for sources with high persistence (p > 0.5) but the biweekly observing
system performs better for sources with low persistence despite its higher detection threshold.
Figure 9. Observing system completeness of a point source imager
as a function of detection threshold and return time. The calculation
is for the ensemble of point sources in Fig. 8. Observing system
completeness for a point source imager is defined here as the ability
to quantify emissions from all point sources larger than 10 kg h−1
.
represent the low tail of the pdf rather than an on/off switch.
The definition of persistence may thus depend on the detection threshold, increasing the importance of that threshold as
a measure of observing system completeness. Further complicating matters is that the instrument detection threshold is
variable, depending notably on the wind speed at the time of
observation. This calls for better characterization of the full
pdf of emissions from point sources as a means to extrapolate
the observations (Allen et al., 2022).
7 Concluding remarks
Satellite observations of atmospheric methane in the shortwave infrared (SWIR) provide an increasingly powerful system for continuous monitoring of emissions from the global
scale down to point sources. We reviewed the current and
scheduled fleet of instruments including area flux mappers to
quantify total emissions on regional scales and point source
imagers to quantify individual source rates. We discussed retrieval methods to infer concentrations from measured radiances, precision and accuracy requirements, inverse methods
to infer emissions from observed concentrations, emission
detection thresholds, and observing system completeness.
Synergy between different satellite instruments is important to exploit. Area flux mappers can constrain total emissions, while point source imagers provide specific facilitylevel attribution. Detection of coarse-resolution hotspots by
area flux mappers can direct targeted observation by point
source imagers to identify the causes (Maasakkers et al.,
2022b). Point source observations with adequate completeness can improve the bottom-up estimates used as prior information in inversions of area flux mapper data. Constellations
of point source imagers can achieve high observing system
completeness in support of point source mapping and leak
detection and repair (LDAR) programs.
Synergy with suborbital (ground-based and airborne) platforms is essential for a multi-tiered observing strategy (Cusworth et al., 2020). Suborbital observations have a unique
role to complement the intrinsic limitations of satellites in
terms of spatial resolution, return time, cloud cover, dark surfaces, and nighttime. Surface measurements are typically 10
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9638 D. J. Jacob et al.: Quantifying methane emissions using satellites
times more sensitive to local emissions than satellite observations (Cusworth et al., 2018). They can also include correlative chemical information such as isotopes, ethane, and
ammonia concentrations (Yuan et al., 2015; Ganesan et al.,
2019; Graven et al., 2019; Pétron et al., 2020; Yang et al.,
2020).
Correlative chemical information available from satellites
needs to be better exploited. Concurrent satellite observations of CO and methane have been used to quantify methane
emissions from open fires (Worden et al., 2013) and from
cities (Plant et al., 2022) by reference to CO emissions, although this is contingent on an accurate CO emission inventory, and errors in these inventories are often large. GeoCarb
will measure methane, CO2, and CO, offering further application of this method, including the use of methane /CO2
enhancement ratios. Concurrent enhancements of CO2 and
methane in oil and gas fields observed by the PRISMA instrument, together with nighttime flare data from the VIIRS
instrument, have been used to identify flaring point sources
and quantify flaring efficiency (Cusworth et al., 2021a). Measurements of ammonia from space (Van Damme et al., 2018)
have the potential to identify livestock sources but have not
yet been used in combination with methane.
Some methane sources are intrinsically difficult to observe
from space, including those over water, the wet tropics, and
the Arctic. Potentially large methane sources over water include offshore oil and gas facilities, wastewater facilities, hydroelectric and agricultural reservoirs, and estuaries. They
can be observed in the sunglint mode or by lidar (Kiemle et
al., 2017; Irakulis-Loitxate et al., 2022b). The wet tropics and
the Arctic are a challenge because of persistent cloudiness,
compounded in the Arctic by high solar zenith angles and polar darkness and by the collocation of oil and gas emissions
with wetland emissions. The MERLIN lidar instrument will
provide a unique observation capability for the Arctic. The
GeoCarb geostationary instrument will increase data density
over tropical South America. The tropics are thought to be
the principal driver for the recent methane increase (Chandra
et al., 2021; Yin et al., 2021; Zhang et al., 2021), and there
would be considerable value in dedicated geostationary or
inclined-orbit satellite observations of the tropics with high
pixel resolution.
The ultimate goal of top-down methane emission estimates is to improve bottom-up estimates, as the latter provide the information needed for climate action by relating
emissions to processes. This calls for partnerships where discrepancies identified by satellite for a particular sector motivate work to improve bottom-up estimates for that sector.
The International Methane Emissions Observatory (IMEO;
United Nations Environmental Program, 2021) aims to facilitate this infusion of top-down information into the improvement of bottom-up inventories on a global scale in support of
the Paris agreement, and initiatives in the oil and gas industry
aim to achieve the same at the level of oil and gas production
fields and individual facilities (Cooper et al., 2022).
The capability is thus emerging for satellite observations
to anchor a global methane monitoring system delivering
global information on emissions in near real time, from the
global scale down to point sources, to support climate policy and to guide corrective action. The basic framework for
building such a facility is already here and will be rapidly
augmented in coming years with the launch of new instruments.
Data availability. The GOSAT methane data in Fig. 2 are
available at https://www.leos.le.ac.uk/data/GHG/GOSAT/v9.
0/CH4_GOS_OCPR_v9.0_final_nceo_2009_2021.tar.gz (last
access: 22 July 2022). The SRON S5P-RemoTeC scientific TROPOMI methane data in Figs. 2 and 3 are available at https://doi.org/10.5281/zenodo.4447228 (Lorente et
al., 2021b). The Sentinel-2 data in Fig. 3 are available at
https://doi.org/10.5270/S2_-d8we2fl (European Space Agency,
2021). The PRISMA and GHGSat data in Fig. 3 are available for
non-commercial uses upon request to the corresponding author. The
data in Figs. 7 and 8 are available at https://doi.org/10.1038/s41586-
019-1720-3 (Duren et al., 2019) for California in 2016–
2017, https://doi.org/10.1021/acs.estlett.1c00173 (Cusworth et al., 2021b) for the Permian Basin in 2019, and
https://doi.org/10.5281/zenodo.5606120 (Cusworth et al., 2021d)
for the AVIRIS-NG/GAO campaigns in 2020–2021.
Author contributions. DJJ wrote the manuscript with contributions from DJV, DHC, PED, CF, RG, LG, JK, JMK, LEO, BP, ZQ,
AKT, JRW, and RMD. DJV, DHC, JK, and ZQ produced the figures. RMD and DHC wrote the initial draft of Sect. 6.2. JK led the
CAMS project that produced this paper.
Competing interests. The contact author has declared that none
of the authors has any competing interests.
Disclaimer. Publisher’s note: Copernicus Publications remains
neutral with regard to jurisdictional claims in published maps and
institutional affiliations.
Acknowledgements. We thank Felipe J. Cardoso-Saldaña and
Cynthia Randles of ExxonMobil Technology and Engineering
Company, Yasjka Meijer and Ben Veihelmann of the ESA,
Ilse Aben of SRON, and Robert Parker of U. Leicester for their valuable comments. We thank Halina Dodd of the Halo Agency, LLC,
for producing Fig. 1. Riley M. Duren and Daniel H. Cusworth acknowledge additional support from Carbon Mapper’s philanthropic
donors. Portions of this research was carried out at the Jet Propulsion Laboratory, California Institute of Technology, under a contract
with the National Aeronautics and Space Administration (grant no.
80NM0018D0004). Philip E. Dennison acknowledges funding from
NASA Carbon Monitoring System (grant no. 80NSSC20K0244).
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9639
Financial support. This research has been supported by the Collaboratory to Advance Methane Science (CAMS) and the National
Aeronautics and Space Administration, Earth Sciences Division
(grant no. NNH20ZDA001N-CMS).
Review statement. This paper was edited by Jason West and reviewed by two anonymous referees.
References
Alexe, M., Bergamaschi, P., Segers, A., Detmers, R., Butz, A.,
Hasekamp, O., Guerlet, S., Parker, R., Boesch, H., Frankenberg,
C., Scheepmaker, R. A., Dlugokencky, E., Sweeney, C., Wofsy,
S. C., and Kort, E. A.: Inverse modelling of CH4 emissions
for 2010–2011 using different satellite retrieval products from
GOSAT and SCIAMACHY, Atmos. Chem. Phys., 15, 113–133,
https://doi.org/10.5194/acp-15-113-2015, 2015.
Allen, D. T., Cardoso-Saldaña, F., and Kimura, Y.: Variability
in spatially and temporally resolved emissions and hydrocarbon source fingerprints for oil and gas sources in shale gas
production regions, Environ. Sci. Technol., 51, 12016–12026,
https://pubs.acs.org/doi/10.1021/acs.est.7b02202, 2017.
Allen, D. T., Cardoso-Saldaña, F. J., Kimura, Y., Chen, Q., Xiang, Z., Zimmerle, D., Bell, C., Lute, C., Duggan, J., and
Harrison, M.: A Methane Emission Estimation Tool (MEET)
for predictions of emissions from upstream oil and gas well
sites with fine scale temporal and spatial resolution: Model
structure and applications, Sci. Total Environ., 829, 154277,
https://doi.org/10.1016/j.scitotenv.2022.154277, 2022.
Angevine, W. M., Peischl, J., Crawford, A., Loughner, C. P.,
Pollack, I. B., and Thompson, C. R.: Errors in top-down estimates of emissions using a known source, Atmos. Chem.
Phys., 20, 11855–11868, https://doi.org/10.5194/acp-20-11855-
2020, 2020.
Ayasse, A. K., Thorpe, A. K., Roberts, D. A., Funk, C. C., Dennison, P. E., Frankenberg, C., Steffke, A., and Aubrey, A. D.:
Evaluating the effects of surface properties on methane retrievals
using a synthetic airborne visible/infrared imaging spectrometer next generation (AVIRIS-NG) image, Remote Sens. Environ.,
215, 386–397, https://doi.org/10.1016/j.rse.2018.06.018, 2018.
Ayasse, A. K., Dennison, P. E., Foote, M., Thorpe, A. K.,
Joshi, S., Green, R. O., Duren, R. M., Thompson, D.
R., and Roberts, D. A.: Methane mapping with future
satellite imaging spectrometers, Remote Sensing, 11, 3054,
https://doi.org/10.3390/rs11243054, 2019.
Barré, J., Aben, I., Agustí-Panareda, A., Balsamo, G., Bousserez,
N., Dueben, P., Engelen, R., Inness, A., Lorente, A., McNorton,
J., Peuch, V.-H., Radnoti, G., and Ribas, R.: Systematic detection
of local CH4 anomalies by combining satellite measurements
with high-resolution forecasts, Atmos. Chem. Phys., 21, 5117–
5136, https://doi.org/10.5194/acp-21-5117-2021, 2021.
Bergamaschi, P., Houweling, S., Segers, A., Krol, M., Frankenberg, C., Scheepmaker, R. A., Dlugokencky, E., Wofsy, S.
C., Kort, E. A., Sweeney, C., Schuck, T., Brenninkmeijer, C.,
Chen, H., Beck, V., and Gerbig, C.: Atmospheric CH4 in
the first decade of the 21st century: Inverse modeling analysis using SCIAMACHY satellite retrievals and NOAA surface measurements, J. Geophys. Res.-Atmos., 118, 7350–7369,
https://doi.org/10.1002/jgrd.50480, 2013.
Bloom, A. A., Bowman, K. W., Lee, M., Turner, A. J., Schroeder,
R., Worden, J. R., Weidner, R., McDonald, K. C., and Jacob, D. J.: A global wetland methane emissions and uncertainty dataset for atmospheric chemical transport models
(WetCHARTs version 1.0), Geosci. Model Dev., 10, 2141–2156,
https://doi.org/10.5194/gmd-10-2141-2017, 2017.
Borchardt, J., Gerilowski, K., Krautwurst, S., Bovensmann, H.,
Thorpe, A. K., Thompson, D. R., Frankenberg, C., Miller, C.
E., Duren, R. M., and Burrows, J. P.: Detection and quantification of CH4 plumes using the WFM-DOAS retrieval on AVIRISNG hyperspectral data, Atmos. Meas. Tech., 14, 1267–1291,
https://doi.org/10.5194/amt-14-1267-2021, 2021.
Bousserez, N., Henze, D.K., Perkins, A., Bowman, K.W., Lee,
M., Liu, J., Deng, F., and Jones, D. B. A.: Improved analysis error covariance matrix for high-dimensional variational inversions: application to source estimation using a 3D atmospheric transport model, Q. J. Roy. Meteor. Soc., 141, 1906–
1921, https://doi.org/10.1002/qj.2495, 2015.
Brandt, A. R., Heath, G. R., and Cooley, D.: Methane
leaks from natural gas systems follow extreme distributions, Environ. Sci. Technol., 50, 12512–12520,
https://doi.org/10.1021/acs.est.6b04303, 2016.
Brasseur, G. P. and Jacob, D. J.: Modeling of Atmospheric
Chemistry, Cambridge University Press, Cambridge, UK,
https://doi.org/10.1017/9781316544754, 2017.
Bruhwiler, L., Basu, S., Butler, J. H., Chatterjee, A., Dlugokencky,
E., Kenney, M. A., McComiskey, A., Montzka, S. A., and Stanitski, D.: Observations of greenhouse gases as climate indicators, Clim. Change, 165, 12, https://doi.org/10.1007/s10584-021-
03001-7, 2021.
Buchwitz, M., Reuter, M., Schneising, O., Boesch, H., Guerlet,
S., Dils, B., Aben, I., Armante, R., Bergamaschi, P., Blumenstock, T., Bovensmann, H., Brunner, D., Buchmann, B., Burrows, J., Butz, A., Chédin, A., Chevallier, F., Crevoisier, C.,
Deutscher, N., Frankenberg, C., Hase, F., Hasekamp, O., Heymann, J., Kaminski, T., Laeng, A., Lichtenberg, G., Mazière,
M. D., Noël, S., Notholt, J., Orphal, J., Popp, C., Parker, R.,
Scholze, M., Sussmann, R., Stiller, G., Warneke, T., Zehner,
C., Bril, A., Crisp, D., Griffith, D., Kuze, A., O’Dell, C.,
Oshchepkov, S., Sherlock, V., Suto, H.,Wennberg, P.,Wunch,
D., Yokota, T., and Yoshida, Y.: The Greenhouse Gas Climate Change Initiative (GHG-CCI): Comparison and quality
assessment of near-surface-sensitive satellite-derived CO2 and
CH4 global data sets, Remote Sens. Environ., 162, 344–362,
https://doi.org/10.1016/j.rse.2013.04.024, 2015.
Butz, A., Galli, A., Hasekamp, O., Landgraf, J., Tol, P., and
Aben, I.: TROPOMI aboard Precursor Sentinel-5 Precursor:
Prospective performance of CH4 retrievals for aerosol and cirrus loaded atmospheres, Remote Sens. Environ., 120, 267–276,
https://doi.org/10.1016/j.rse.2011.05.030, 2012.
Cardoso-Saldaña, F. J., and Allen, D. T.: Projecting the temporal evolution of methane emissions from oil and gas
production sites, Environ. Sci. Technol., 54, 14172–14181,
https://doi.org/10.1021/acs.est.0c03049, 2020.
Chandra, N., Patra, P.K., Bisht, J. S. H., Ito, A., Umezawa, T., Saigusa, N., Morimoto, S., Aoki, S., Janssens-Maenhout, G., Fujita, R., Takigawa, M., Watanabe, and Saitoh, N.: Emissions from
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9640 D. J. Jacob et al.: Quantifying methane emissions using satellites
the oil and gas sectors, coal mining and ruminant farming drive
methane growth over the past three decades, J. Met. Soc. Jpn.,
99, 309–337, https://doi.org/10.2151/jmsj.2021-015, 2021.
Chen, Y., Sherwin, E.D., Berman, E. S. F., Jones, B. B., Gordon,
M. P., Wetherley, E. B., Kort, E. A., and Brandt, A. R.: Quantifying regional methane emissions in the New MexicoPermian
Basin with a comprehensive aerial survey, Environ. Sci. Technol.,
https://doi.org/10.1021/acs.est.1c06458, 2022.
Chen, Z., Jacob, D., Nesser, H., Sulprizio, M., Lorente, A., Varon,
D., Lu, X., Shen, L., Qu, Z., Penn, E., and Yu, X.: Methane
emissions from China: a high-resolution inversion of TROPOMI
satellite observations, Atmos. Chem. Phys. Discuss. [preprint],
https://doi.org/10.5194/acp-2022-303, in review, 2022.
Cho, T., Chung, J., Miller, S. M., and Saibaba, A. K.: Computationally efficient methods for large-scale atmospheric
inverse modeling, Geosci. Model Dev., 15, 5547–5565,
https://doi.org/10.5194/gmd-15-5547-2022, 2022.
Cooper, J., Dubey, L., and Hawkes, A.: Methane detection and
quantification in the upstream oil and gas sector: the role of
satellites in emissions detection, reconciling and reporting, Environ. Sci. Atmos., 2, 9–23, https://doi.org/10.1039/D1EA00046B,
2022.
Cusworth, D. H., Jacob, D. J., Sheng, J.-X., Benmergui, J., Turner,
A. J., Brandman, J., White, L., and Randles, C. A.: Detecting high-emitting methane sources in oil/gas fields using
satellite observations, Atmos. Chem. Phys., 18, 16885–16896,
https://doi.org/10.5194/acp-18-16885-2018, 2018.
Cusworth, D. H., Jacob, D. J., Varon, D. J., Chan Miller, C.,
Liu, X., Chance, K., Thorpe, A. K., Duren, R. M., Miller, C.
E., Thompson, D. R., Frankenberg, C., Guanter, L., and Randles, C. A.: Potential of next-generation imaging spectrometers
to detect and quantify methane point sources from space, Atmos. Meas. Tech., 12, 5655–5668, https://doi.org/10.5194/amt12-5655-2019, 2019.
Cusworth, D. H., Duren, R. M., Yadav, V., Thorpe, A. K., Verhulst,
K., Sander, S., Hopkins, F., Raiq, T., and Miller, C. E.: Synthesis of methane observations across scales: Strategies for deploying a multitiered observing network, Geophys. Res. Lett., 47,
e2020GL087869, https://doi.org/10.1029/2020GL087869, 2020.
Cusworth, D. H., Duren, R., Thorpe, A. K., Pandey, S., Aben,
I., Jervis, D., Varon, D., Jacob, D. J., Randles, C. A., Gautam, R., Omara, M., Schade, G., Dennison, P. E., Frankenberg, C., Gordon, D., Lopinto, E., and Miller, C. E.: Multisatellite imaging of a gas well blowout provides new insights for
methane monitoring, Geophys. Res. Lett., 48, e2020GL090864,
https://doi.org/10.1029/2020GL090864, 2021a.
Cusworth, D. H., Duren, R. M., Thorpe, A. K., Olson-Duvall,
W., Heckler, J., Chapman, J. W., Eastwod, M. L., Helmlinger, M. C., Green, R. O., Asner, G. P., Dennison, P. E.,
and Miller, C. E.: Intermittency of large methane emitters in
the Permian Basin, Environ. Sci. Technol. Lett., 8, 567–573,
https://doi.org/10.1021/acs.estlett.1c00173, 2021b.
Cusworth, D. H., Bloom, A. A., Ma, S., Miller, C. E., Bowman, K., Yin, Y., Maasakkers, J. D., Zhang, Y., Scarpelli,
T. R., Qu, Z., Jacob, D. J. and Worden, J. R.: A Bayesian
framework for deriving sector-based methane emissions from
top-down fluxes, Nat. Commun. Earth Environ., 2, 242,
https://doi.org/10.1038/s43247-021-00312-6, 2021c.
Cusworth, D. H., Duren, R., Ayasse, A., Thorpe, A., Asner, G., and Green, R.: Methane plumes from airborne surveys 2020–2021 (1.0), Zenodo [data set],
https://doi.org/10.5281/zenodo.5606120, 2021d.
Cusworth, D. H., Thorpe, A. K., Ayasse, A. K., Stepp, D., Heckler,
J., Asner, G. P., Miller, C. E., Chapman, J. W., Eastwood, M.
L., Green, R. O., Hmiel, B., Lyon, D., and Duren, R. M.: Strong
methane point sources contribute a disproportionate fraction of
total emissions across multiple basins in the U.S., Earth ArXiv,
https://doi.org/10.31223/X53P88, 2022.
Deng, Z., Ciais, P., Tzompa-Sosa, Z. A., Saunois, M., Qiu, C., Tan,
C., Sun, T., Ke, P., Cui, Y., Tanaka, K., Lin, X., Thompson, R.
L., Tian, H., Yao, Y., Huang, Y., Lauerwald, R., Jain, A. K., Xu,
X., Bastos, A., Sitch, S., Palmer, P. I., Lauvaux, T., d’Aspremont,
A., Giron, C., Benoit, A., Poulter, B., Chang, J., Petrescu, A. M.
R., Davis, S. J., Liu, Z., Grassi, G., Albergel, C., Tubiello, F. N.,
Perugini, L., Peters, W., and Chevallier, F.: Comparing national
greenhouse gas budgets reported in UNFCCC inventories against
atmospheric inversions, Earth Syst. Sci. Data, 14, 1639–1675,
https://doi.org/10.5194/essd-14-1639-2022, 2022.
Duren, R. M., Thorpe, A. K., Foster, K. T., Rafiq, T., Hopkins, F. M.,
Yadav, V., Bue, B. D., Thompson, D. R., Conley, S., Colombi,
N. K., Frankenberg, C., McCubbin, I.B., Eastwood, M. L., Falk,
M., Herner, J. D., Croes, B. E., Green, R. O., and Miller, C.
E.: California’s methane super-emitters, Nature, 575, 180–184,
https://doi.org/10.1038/s41586-019-1720-3, 2019.
Duren, R. M.: Carbon Mapper: on-orbit performance predictions
and airborne prototyping, American Geophysical Union Fall
Meeting, New Orleans, 13–17 December 2021, A53F-05, https:
//agu.confex.com/agu/fm21/meetingapp.cgi/Paper/985888 (last
access: 23 July 2022), 2021.
Ehret, T., Bousquet, P., Pierangelo, C., Alpers M., Millet, B., Abshire, J.B., Bovensmann, H., Burrows, J.P., Chevallier, F., Ciais,
P., Crevoisier, C., Fix, A., Flamant, P., Frankenberg, C., Gibert,
F., Heim, B., Heimann, M., Houweling, S., Hubberten, H.W.,
Jöckel, P., Law, K., Löw, A., Marshall, J., Agusti-Panareda,
A., Payan, S., Prigent, C., Rairoux, P., Sachs, T., Scholze, M.,
and Wirth, M.: MERLIN: A French-German space lidar mission dedicated to atmospheric methane, Remote Sens., 9, 1052,
https://doi.org/10.3390/rs9101052, 2017.
Ehret, T., De Truchis, A., Mazzolini, M., Morel, J.-M.,
d’Aspremont, A., Lauvaux, T., and Facciolo, G.: Global tracking and quantification of oil and gas methane leaks from recurrent Sentinel-2 imagery, arXiv, 2110.11832v1, physics.ao-ph,
https://doi.org/10.48550/arXiv.2110.11832, 2022.
European Space Agency: Sentinel-5, ESA SP-1336,
ISBN 978-92-9221-434-0, https://esamultimedia.esa.int/docs/
EarthObservation/SP_1336_Sentinel-5_web.pdf (last access: 23
July 2022), 2020.
European Space Agency: Copernicus Sentinel-2 (processed by
ESA), MSI Level-1C TOA Reflectance Product, Collection 0,
European Space Agency [data set], https://doi.org/10.5270/S2_-
d8we2fl, 2021.
Feng, L., Palmer, P. I., Bösch, H., Parker, R. J., Webb, A. J., Correia, C. S. C., Deutscher, N. M., Domingues, L. G., Feist, D. G.,
Gatti, L. V., Gloor, E., Hase, F., Kivi, R., Liu, Y., Miller, J. B.,
Morino, I., Sussmann, R., Strong, K., Uchino, O., Wang, J., and
Zahn, A.: Consistent regional fluxes of CH4 and CO2 inferred
from GOSAT proxy XCH4 : XCO2 retrievals, 2010–2014, AtAtmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9641
mos. Chem. Phys., 17, 4781–4797, https://doi.org/10.5194/acp17-4781-2017, 2017.
Foote, M. D., Dennison, P. E., Thorpe, A. K., Thompson, D. R., Jongaramrungruang, S., Frankenberg, C., and
Joshi, S. C.: Fast and accurate retrieval of methane concentration from imaging spectrometer data using sparsity
prior, IEEE Trans. Geosci. Remote Sens., 58, 6480–6492,
https://doi.org/10.1109/TGRS.2020.2976888, 2020.
Fox, T. A., Gao, M., Barchyn, T. E., Jamin, Y. L., and
Hugenholtz, C. H.: An agent-based model for estimating emissions reduction equivalence among leak detection
and repair programs, J. Cleaner Production, 282, 125237,
https://doi.org/10.1016/j.jclepro.2020.125237, 2021.
Frankenberg, C., Platt, U., and Wagner, T.: Iterative maximum
a posteriori (IMAP)-DOAS for retrieval of strongly absorbing trace gases: Model studies for CH4 and CO2 retrieval
from near infrared spectra of SCIAMACHY onboard ENVISAT,
Atmos. Chem. Phys., 5, 9–22, https://doi.org/10.5194/acp-5-9-
2005, 2005.
Frankenberg, C., Thorpe, A. K., Thompson, D. R., Hulley,
G., Kort, E. A., Vance, N., Borchardt, J., Krings, T., Gerilowski, K., Sweeney, C., Conley, S., Bue, B. D., Aubrey,
A. D., Hook, S., Green, R. O.: Airborne methane remote
measurements reveal heavy-tail flux distribution in Four Corners region, P. Natl. Acad. Sci. USA, 113, 9734–9739,
https://doi.org/10.1073/pnas.1605617113, 2016.
Ganesan, A. L., Schwietzke, S., Poulter, B., Arnold, T., Lan, X.,
Rigby, M., Vogel, F. R., van der Werf, G. R., Janssens-Maenhout, G., Boesch, H., Pandey, S., Manning, A. J., Jackson,
R. B.,Nisbet, E. G., and Manning, M. R.: Advancing scientific understanding of the global methane budget in support of
the Paris Agreement, Global Biogeochem. Cy., 33, 1475–1512,
https://doi.org/10.1029/2018GB006065, 2019.
Gauthier, J.-F.: The importance of matching needs to satellite system capability when monitoring methane emissions
from space, IEEE International Geoscience and Remote
Sensing Symposium (IGARSS), Brussels, 11–16 July 2021,
https://doi.org/10.1109/IGARSS47720.2021.9555123, 2021.
GEO, ClimateTRACE, WGIC: GHG Monitoring
from Space: A mapping of capabilities across
public, private, and hybrid satellite missions,
https://wgicouncil.org/publication/reports/industry-reports/
(last access: 23 July 2022), 2021.
Graven, H., Hocking, T., and Zazzeri, G.: Detection of
fossil and biogenic methane at regional scales using
atmospheric radiocarbon, Earth’s Future, 7, 283–299,
https://doi.org/10.1029/2018EF001064, 2019.
Guanter, L. Irakulis-Loitxate, I., Gorroño, J., Sánchez-García, E.,
Cusworth, D. H., Varon, D. J., Cogliati, S., and Colombo, R.:
Mapping methane point emissions with the PRISMA spaceborne
imaging spectrometer, Remote Sens. Environ., 265, 112671,
https://doi.org/10.1016/j.rse.2021.112671, 2021.
Hajny, K. D., Salmon, O. E., Rudek, J., Lyon, D. R., Stuff, A. A.,
Stirm, B. H., Kaeser, R., Floerchinger, C. R., Conley, S., Smith,
M. L., and Shepson, P. B.: Observations of methane emissions
from natural gas-fired power plants, Environ. Sci. Technol., 15,
8976–8984, https://doi.org/10.1021/acs.est.9b01875, 2019.
Heald, C. L., Jacob, D. J., Jones, D. B. A., Palmer, P. I., Logan, J.
A., Streets, D. G., Sachse, G. W., Gille, J. C., Hoffman, R. N.,
and Nehrkorn, T.: Comparative inverse analysis of satellite (MOPITT) and aircraft (TRACE-P) observations to estimate Asian
sources of carbon monoxide, J. Geophys. Res., 109, D23306,
https://doi.org/10.1029/2004JD005185, 2004.
Hill, T. and Nassar, R.: Pixel size and revisit rate requirements
for monitoring power plant CO2 emissions from space, Remote
Sens., 11, 1608, https://doi.org/10.3390/rs11131608, 2019.
Hulley, G. C., Duren, R. M., Hopkins, F. M., Hook, S. J., Vance,
N., Guillevic, P., Johnson, W. R., Eng, B. T., Mihaly, J. M., Jovanovic, V. M., Chazanoff, S. L., Staniszewski, Z. K., Kuai, L.,
Worden, J., Frankenberg, C., Rivera, G., Aubrey, A. D., Miller,
C. E., Malakar, N. K., Sánchez Tomás, J. M., and Holmes,
K. T.: High spatial resolution imaging of methane and other
trace gases with the airborne Hyperspectral Thermal Emission
Spectrometer (HyTES), Atmos. Meas. Tech., 9, 2393–2408,
https://doi.org/10.5194/amt-9-2393-2016, 2016.
IPCC: 2019 Refinement to the 2006 IPCC Guidelines for National
Greenhouse Gas Inventories, edited by: Calvo Buendia, E., Tanabe, K., Kranjc, A., Jamsranjav, B., Fukuda, M., Ngarize, S., Osako, A., Pyrozhenko, Y., Shermanau, P., and Federici, S., IPCC,
Switzerland, https://www.ipcc.ch/report/2019-refinement-to-the2006-ipcc-guidelines (last access: 23 July 2022), 2019.
Irakulis-Loitxate, I., Guanter, L., Liu, Y.-N., Varon, D. J.,
Maasakkers, J. D., Zhang, Y., Thorpe, A. K., Duren, R.
M., Frankenberg, C., Lyon, D., Hmiel, B., Cusworth, D. H.,
Zhang, Y., Segl, K., Gorron, J., Sanchez-Garcıa, E., Sulprizio, M. P., Cao, K., Zhu, H., Liang, J., Li, X., Aben, I.,
and Jacob, D. J.: Satellite-based survey of extreme methane
emissions in the Permian Basin , Sci. Adv., 7, eabf4507,
https://doi.org/10.1126/sciadv.abf4507, 2021.
Irakulis-Loitxate, I., Guanter, L., Maasakkers, J. D., ZavalaAraiza, D., and Aben, I.: Satellites detect abatable
super-emissions in one of the world’s largest methane
hotspot regions, Environ. Sci. Technol. 56, 4, 2143–2152,
https://doi.org/10.1021/acs.est.1c04873, 2022a.
Irakulis-Loitxate, I., Gorrono, J., Zavala-Araiza, D., and Guanter,
L.: Satellites detect a methane ultra-emission event from an offshore platform in the Gulf of Mexico, Environ. Sci. Technol.
Lett., 9, 522-525, https://doi.org/10.1021/acs.estlett.2c00225,
2022b.
Jacob, D. J., Turner, A. J., Maasakkers, J. D., Sheng, J., Sun,
K., Liu, X., Chance, K., Aben, I., McKeever, J., and Frankenberg, C.: Satellite observations of atmospheric methane and
their value for quantifying methane emissions, Atmos. Chem.
Phys., 16, 14371–14396, https://doi.org/10.5194/acp-16-14371-
2016, 2016.
Janardanan, R., Maksyutov, S., Tsuruta, A., Wang, F., Tiwari, Y.
K., Valsala, V., Ito, A., Yoshida, Y., Kaiser, J. W., JanssensMaenhout, G., Arshinov, M., Sasakawa, M., Tohjima, Y., Worthy, D. E. J., Dlugokencky, E. J., Ramonet, M., Arduini, J.,
Lavric, J. V., Piacentino, S., Krummel, P. B., Langenfelds, R.
L., Mammarella, I., and Matsunaga, T.: Country-scale analysis
of methane emissions with a high-tesolution inverse model using GOSAT and surface observations, Remote Sens., 12, 375,
https://doi.org/10.3390/rs12030375, 2020.
Jervis, D., McKeever, J., Durak, B. O. A., Sloan, J. J., Gains,
D., Varon, D. J., Ramier, A., Strupler, M., and Tarrant, E.:
The GHGSat-D imaging spectrometer, Atmos. Meas. Tech., 14,
2127–2140, https://doi.org/10.5194/amt-14-2127-2021, 2021.
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9642 D. J. Jacob et al.: Quantifying methane emissions using satellites
Johnson, D. and Heltzel, R.: On the long-term temporal variations in methane emissions from an unconventional natural gas well site, ACS Omega, 6, 14200–14207,
https://doi.org/10.1021/acsomega.1c00874, 2021.
Jongaramrungruang, S., Frankenberg, C., Matheou, G., Thorpe,
A. K., Thompson, D. R., Kuai, L., and Duren, R. M.: Towards accurate methane point-source quantification from highresolution 2-D plume imagery, Atmos. Meas. Tech., 12, 6667–
6681, https://doi.org/10.5194/amt-12-6667-2019, 2019.
Jongaramrungruang, S., Matheou, G., Thorpe, A. K., Zeng, Z.-C.,
and Frankenberg, C.: Remote sensing of methane plumes: instrument tradeoff analysis for detecting and quantifying local
sources at global scale, Atmos. Meas. Tech., 14, 7999–8017,
https://doi.org/10.5194/amt-14-7999-2021, 2021.
Jongaramrungruang, S., Matheou, G., Thorpe, A. K., Zeng, Z.-
C., and Frankenberg, C.: MethaNet – An AI-driven approach to
quantifying methane point-source emission from high-resolution
2-D plume imagery, Remote Sens. Environ., 269, 112809,
https://doi.org/10.1016/j.rse.2021.112809, 2022.
Karion, A., Sweeney, C., Kort, E. A., Shepson, P. B., Brewer, A.,
Cambaliza, M., Conley, S. A., Davis, K., Deng, A., Hardesty, M.,
Herndon, S. C., Lauvaux, T., Lavoie, T., Lyon, D., Newberger, T.,
Petron, G., Rella, C., Smith, M., Wolter, S., Yacovitch, T. I., and
Tans, P.: Aircraft-Based Estimate of Total Methane Emissions
from the Barnett Shale Region, Environ. Sci. Technol., 49, 8124–
8131, https://doi.org/10.1021/acs.est.5b00217, 2015.
Kemp, C. E., Ravikumar, A. P., and Brandt, A. R.: Comparing natural gas leakage detection technologies using an open-source “virtual gas field” simulator, Environ. Sci. Technol., 50, 4546–4553,
https://doi.org/10.1021/acs.est.5b06068, 2016.
Kiemle C., Ehret, G., Amediek, A., Fix, A., Quatrevalet, M., and
Wirth, M.: Potential of spaceborne lidar measurements of carbon dioxide and methane emissions from strong point sources,
Remote Sens., 9, 1137, https://doi.org/10.3390/rs9111137, 2017.
Koo, J.-H., Walker, K. A., Jones, A., Sheese, P. E., Boone, C. D.,
Bernath, P. F., and Manney, G. L.: Global climatology based on
the ACE-FTS version 3.5 dataset: Addition of mesospheric levels
and carbon-containing species in the UTLS, J. Quant. Spectrosc.
Ra., 186, 52–62, https://doi.org/10.1016/j.jqsrt.2016.07.003,
2017.
Krings, T., Gerilowski, K., Buchwitz, M., Reuter, M., Tretner, A., Erzinger, J., Heinze, D., Pflüger, U., Burrows, J. P.,
and Bovensmann, H.: MAMAP – a new spectrometer system for column-averaged methane and carbon dioxide observations from aircraft: retrieval algorithm and first inversions for
point source emission rates, Atmos. Meas. Tech., 4, 1735–1758,
https://doi.org/10.5194/amt-4-1735-2011, 2011.
Lan, X., Basu, S., Schwietzke, S., Bruhwiler, L. M. P., Dlugokencky, E. J., Michel, S. E., et al.: Improved constraints on global methane emissions and sinks using
δ13C-CH4, Glob. Biogeochem. Cy., 35, e2021GB007000,
https://doi.org/10.1029/2021GB007000, 2021.
Lauvaux, T., Giron, C., Mazzolini, M., d’Aspremont, A., Duren,
R., Cusworth, D., Shindell, D., and Ciais, P.: Global assessment
of oil and gas methane ultra-emitters, Science, 375, 557–561,
https://doi.org/10.31223/X5NS54, 2022.
Lorente, A., Borsdorff, T., Butz, A., Hasekamp, O., aan de Brugh,
J., Schneider, A., Wu, L., Hase, F., Kivi, R., Wunch, D., Pollard,
D. F., Shiomi, K., Deutscher, N. M., Velazco, V. A., Roehl, C.
M., Wennberg, P. O., Warneke, T., and Landgraf, J.: Methane
retrieved from TROPOMI: improvement of the data product
and validation of the first 2 years of measurements, Atmos.
Meas. Tech., 14, 665–684, https://doi.org/10.5194/amt-14-665-
2021, 2021a.
Lorente, A., Borsdorff, T., aan de Brugh, J., Landgraf,
J., and Hasekamp, O.: SRON S5P – RemoTeC scientific TROPOMI XCH4 dataset, Zenodo [data set],
https://doi.org/10.5281/zenodo.4447228, 2021b.
Lu, X., Jacob, D. J., Zhang, Y., Maasakkers, J. D., Sulprizio, M. P.,
Shen, L., Qu, Z., Scarpelli, T. R., Nesser, H., Yantosca, R. M.,
Sheng, J., Andrews, A., Parker, R. J., Boesch, H., Bloom, A. A.,
and Ma, S.: Global methane budget and trend, 2010–2017: complementarity of inverse analyses using in situ (GLOBALVIEWplus CH4 ObsPack) and satellite (GOSAT) observations, Atmos. Chem. Phys., 21, 4637–4657, https://doi.org/10.5194/acp21-4637-2021, 2021.
Lu, X., Jacob, D. J., Wang, H., Maasakkers, J. D., Zhang, Y.,
Scarpelli, T. R., Shen, L., Qu, Z., Sulprizio, M. P., Nesser, H.,
Bloom, A. A., Ma, S., Worden, J. R., Fan, S., Parker, R. J.,
Boesch, H., Gautam, R., Gordon, D., Moran, M. D., Reuland,
F., Villasana, C. A. O., and Andrews, A.: Methane emissions in
the United States, Canada, and Mexico: evaluation of national
methane emission inventories and 2010–2017 sectoral trends
by inverse analysis of in situ (GLOBALVIEWplus CH4 ObsPack) and satellite (GOSAT) atmospheric observations, Atmos.
Chem. Phys., 22, 395–418, https://doi.org/10.5194/acp-22-395-
2022, 2022.
Lyon, D. R., Zavala-Araiza, D., Alvarez, R. A., Harriss, R., Palacios, V., Lan, X., Talbot, R., Lavoie, T., Shepson, P., Yacovitch, T. I., Herndon, S. C., Marchese, A. J., Zimmerle,
D., Robinson, A. L., and Hamburg, S. P.: Constructing a
spatially resolved methane emission inventory for the Barnett Shale region, Environ. Sci. Technol., 49, 8147–8157,
https://doi.org/10.1021/es506359c, 2015.
Lyon, D. R., Alvarez, R. A., Zavala-Araiza, D., Brandt,
A. R., Jackson, R. B., and Hamburg, S. P.: Aerial surveys of elevated hydrocarbon emissions from oil and gas
production sites, Environ. Sci. Technol., 50, 4877–4886,
https://doi.org/10.1021/acs.est.6b00705, 2016.
Lyon, D. R., Hmiel, B., Gautam, R., Omara, M., Roberts, K. A.,
Barkley, Z. R., Davis, K. J., Miles, N. L., Monteiro, V. C.,
Richardson, S. J., Conley, S., Smith, M. L., Jacob, D. J., Shen,
L., Varon, D. J., Deng, A., Rudelis, X., Sharma, N., Story, K.
T., Brandt, A. R., Kang, M., Kort, E. A., Marchese, A. J., and
Hamburg, S. P.: Concurrent variation in oil and gas methane
emissions and oil price during the COVID-19 pandemic, Atmos. Chem. Phys., 21, 6605–6626, https://doi.org/10.5194/acp21-6605-2021, 2021.
Maasakkers, J. D., Jacob, D. J., Sulprizio, M. P., Turner, A. J.,
Weitz, M., Wirth, T., Hight, C., DeFigueiredo, M., Desai, M.,
Schmeltz, R., Hockstad, L., Bloom, A. A., Bowman, K. W.,
Jeong, S., and Fischer, M. L.: Gridded national inventory of U.
S. methane emissions, Environ. Sci. Technol., 50, 13123–13133,
https://doi.org/10.1021/acs.est.6b02878, 2016.
Maasakkers, J. D., Jacob, D. J., Sulprizio, M. P., Scarpelli, T. R.,
Nesser, H., Sheng, J.-X., Zhang, Y., Hersher, M., Bloom, A.
A., Bowman, K. W., Worden, J. R., Janssens-Maenhout, G., and
Parker, R. J.: Global distribution of methane emissions, emisAtmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9643
sion trends, and OH concentrations and trends inferred from
an inversion of GOSAT satellite data for 2010–2015, Atmos.
Chem. Phys., 19, 7859–7881, https://doi.org/10.5194/acp-19-
7859-2019, 2019.
Maasakkers, J. D., Jacob, D. J., Sulprizio, M. P., Scarpelli, T.
R., Nesser, H., Sheng, J., Zhang, Y., Lu, X., Bloom, A. A.,
Bowman, K. W., Worden, J. R., and Parker, R. J.: 2010–2015
North American methane emissions, sectoral contributions, and
trends: a high-resolution inversion of GOSAT observations of
atmospheric methane, Atmos. Chem. Phys., 21, 4339–4356,
https://doi.org/10.5194/acp-21-4339-2021, 2021.
Maasakkers, J. D., Omara, M., Gautam, R., Lorente, A., Pandey,
S., Toi, P., Borsdorff, T., Houweling, S., and Aben, I.: Reconstructing and quantifying methane emissions from the full
duration of a 38-day natural gas well blowout using spacebased observations, Remote Sens. Environ., 270, 112755,
https://doi.org/10.1016/j.rse.2021.112755, 2022a.
Maasakkers, J. D., Varon, D. J., Elfarsdottir, A., McKeever, J.
D., Jervis, D., Mahapatra, G., Pandey, S., Lorente, A., Borsdorff, T., Foorthuis, L. R., Schuit, B. J., Tol, P., van Kempen, T. A., van Hees, R., and Aben, I.: Using satellites to uncover large methane emissions from landfills, arXiv [preprint],
https://doi.org/10.31223/X5N33G, 2022b.
MacLean, J.-P.: Detecting and quantifying methane emissions with
the high-resolution GHGSat satellite constellation, American
Geophysical Union Fall meeting, New Orleans, 13–17 December
2021, A54F-01, https://agu.confex.com/agu/fm21/meetingapp.
cgi/Paper/946992 (last access: 23 July 2022), 2021.
Miller, S. M. and Michalak, A. M.: Constraining sector-specific
CO2 and CH4 emissions in the US, Atmos. Chem. Phys., 17,
3963–3985, https://doi.org/10.5194/acp-17-3963-2017, 2017.
Miller, S. M., Michalak, A. M., Detmers, R. G., Hasekamp, O.
P., Bruhwiler, L. M. P., and Schwietzke, S.: China’s coal mine
methane regulations have not curbed growing emissions, Nat.
Commun., 10, 303, https://doi.org/10.1038/s41467-018-07891-
7, 865, 2019.
Miller, S. M., Saibaba, A. K., Trudeau, M. E., Mountain, M. E.,
and Andrews, A. E.: Geostatistical inverse modeling with very
large datasets: an example from the Orbiting Carbon Observatory 2 (OCO-2) satellite, Geosci. Model Dev., 13, 1771–1785,
https://doi.org/10.5194/gmd-13-1771-2020, 2020.
Moore, B., Crowell, S. M. R., Rayner, P. J., Kumer, J., O’Dell,
C. W., O’Brien, D., Utembe, S., Polonsky, I., Schimel, D., and
Lemen, J.: The Potential of the Geostationary Carbon Cycle
Observatory (GeoCarb) to provide multi-scale constraints on
the carbon cycle in the Americas, Front. Environ. Sci., 6, 109,
https://doi.org/10.3389/fenvs.2018.00109, 2018.
Naik, V., Szopa, S., Adhikary, B., Artaxo, P., Berntsen, T.,
Collins, W. D., Fuzzi, S., Gallardo, L., Kiendler Scharr, A.,
Klimont, Z., Liao, H., Unger, N., and Zanis, P.: Short-Lived
Climate Forcers, in: Climate Change 2021: The Physical Science Basis. Contribution of Working Group I to the Sixth
Assessment Report of the Intergovernmental Panel on Climate Change, edited by: Masson-Delmotte, V., Zhai, P., Pirani, A., Connors, S. L., Péan, C., Berger, S., Caud, N.,
Chen, Y., Goldfarb, L., Gomis, M. L., Huang, M., Leitzell,
K., Lonnoy, E., Matthews, J. B. R., Maycock, T. K., Waterfield, T., Yelekçi, O., Yu, R., and Zhou, B., Cambridge University Press, https://www.ipcc.ch/report/ar6/wg1/downloads/
report/IPCC_AR6_WGI_Chapter06.pdf (last access: 23 July
2022), 2021.
Nesme N, Marion, R., Lezeaux, O., Doz, S., Camy-Peyret, C., and
Foucher, P.-Y.: Joint use of in-scene background radiance estimation and optimal estimation methods for quantifying methane
emissions using PRISMA hyperspectral satellite data: application to the Korpezhe industrial site, Remote Sens., 13, 4992,
https://doi.org/10.3390/rs13244992, 2021.
Nesser, H., Jacob, D. J., Maasakkers, J. D., Scarpelli, T. R., Sulprizio, M. P., Zhang, Y., and Rycroft, C. H.: Reduced-cost
construction of Jacobian matrices for high-resolution inversions of satellite observations of atmospheric composition, Atmos. Meas. Tech., 14, 5521–5534, https://doi.org/10.5194/amt14-5521-2021, 2021.
NIES: GOSAT-GW mission: background, aims, and mission
requirements, https://gosat-gw.global-atmos-chem-lab.jp/en/
collaboration/ (last access: 23 July 2022), 2021.
Nisbet, E. G., Fisher, R. E., Lowry, D., France, J. L., Allen, G.,
Bakkaloglu, S., Broderick, T. J., Cain, M., Coleman, M., Fernandez, J., Forster, G., Griffiths, P. T., Iverach, C. P., Kelly, B. F. J.,
Manning, M. R., Nisbet-Jones, P. B. R., Pyle, J. A., TownsendSmall, A., al-Shalaan, A., Warwick, N., and Zazzeri, G.:
Methane mitigation: methods to reduce emissions, on the path
to the Paris agreement, Rev. Geophys., 58, e2019RG000675,
https://doi.org/10.1029/2019RG000675, 2020.
Noël, S., Reuter, M., Buchwitz, M., Borchardt, J., Hilker, M.,
Schneising, O., Bovensmann, H., Burrows, J. P., Di Noia, A.,
Parker, R. J., Suto, H., Yoshida, Y., Buschmann, M., Deutscher,
N. M., Feist, D. G., Griffith, D. W. T., Hase, F., Kivi, R., Liu,
C., Morino, I., Notholt, J., Oh, Y.-S., Ohyama, H., Petri, C., Pollard, D. F., Rettinger, M., Roehl, C., Rousogenous, C., Sha, M.
K., Shiomi, K., Strong, K., Sussmann, R., Té, Y., Velazco, V. A.,
Vrekoussis, M., and Warneke, T.: Retrieval of greenhouse gases
from GOSAT and GOSAT-2 using the FOCAL algorithm, Atmos. Meas. Tech., 15, 3401–3437, https://doi.org/10.5194/amt15-3401-2022, 2022.
Omara, M., Zavala-Araiza, D., Lyon, D. R., Hmiel, B., Roberts, K.
A., and Hamburg, S. P.: Methane emissions from US low production oil and natural gas well sites, Nat. Commun., 13, 2085,
https://doi.org/10.1038/s41467-022-29709-3, 2022.
Palmer, P. I., Feng, L., Lunt, M. F., Parker, R. J., Bosch, H., Lan,
X., Lorente, A., and Borsdorff, T.: The added value of satellite observations of methane for understanding the contemporary methane budget, Phil. Trans. R. Soc. A, 379, 20210106,
https://doi.org/10.1098/rsta.2021.0106, 2021.
Pandey, S., Gautam, R., Houweling, S., Denier van der Gon,
H., Sadavarte, P., Borsdorff, T., Hasekamp, O., Landgraf, J.,
Tol, P., van Kempen, T., Hoogeveen, R., van Hees, R., Hamburg, S. P., Maasakkers, J. D., and Aben, I.: Satellite observations reveal extreme methane leakage from a natural gas
well blowout, P. Natl. Acad. Sci. USA, 116, 23676–23681,
https://doi.org/10.1073/pnas.1908712116, 2019.
Parker, R. J., Webb, A., Boesch, H., Somkuti, P., Barrio Guillo,
R., Di Noia, A., Kalaitzi, N., Anand, J. S., Bergamaschi, P.,
Chevallier, F., Palmer, P. I., Feng, L., Deutscher, N. M., Feist,
D. G., Griffith, D. W. T., Hase, F., Kivi, R., Morino, I., Notholt,
J., Oh, Y.-S., Ohyama, H., Petri, C., Pollard, D. F., Roehl, C.,
Sha, M. K., Shiomi, K., Strong, K., Sussmann, R., Té, Y., Velazco, V. A., Warneke, T., Wennberg, P. O., and Wunch, D.: A
https://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9644 D. J. Jacob et al.: Quantifying methane emissions using satellites
decade of GOSAT Proxy satellite CH4 observations, Earth Syst.
Sci. Data, 12, 3383–3412, https://doi.org/10.5194/essd-12-3383-
2020, 2020.
Pétron, G., Miller, B., Vaughn, B., Thorley, E., Kofler, J., MielkeMaday, I., Sherwood, O., Dlugokencky, E., Hall, B., Schwietzke, S., Conley, S., Peischl, J., Lang, P., Moglia, E., Crotwell,
M., Crotwell, A., Sweeney, C., Newberger, T., Wolter, S., Kitzis,
D., Bianco, L., King, C., Coleman, T., White, A., Rhodes, M.,
Tans, P., and Schnell, R.: Investigating large methane enhancements in the U.S. San Juan Basin, Elem. Sci. Anth, 8, 038,
https://doi.org/10.1525/elementa.038, 2020.
Plant, G., Kort, E. A., Murray, L. T., Maasakkers, J. D.,
and Aben, I.: Evaluating urban methane emissions from
space using TROPOMI methane and carbon monoxide observations, Remote Sens. Environ., 268, 112756,
https://doi.org/10.1016/j.rse.2021.112756, 2022.
Prather, M. J., Holmes, C. D., and Hsu, J.: Reactive greenhouse
gas scenarios: Systematic exploration of uncertainties and the
role of atmospheric chemistry, Geophys. Res. Lett., 39, L09803,
https://doi.org/10.1029/2012gl051440, 2012.
Qu, Z., Jacob, D. J., Shen, L., Lu, X., Zhang, Y., Scarpelli, T.
R., Nesser, H., Sulprizio, M. P., Maasakkers, J. D., Bloom,
A. A., Worden, J. R., Parker, R. J., and Delgado, A. L.:
Global distribution of methane emissions: a comparative inverse analysis of observations from the TROPOMI and GOSAT
satellite instruments, Atmos. Chem. Phys., 21, 14159–14175,
https://doi.org/10.5194/acp-21-14159-2021, 2021.
Riris, H., Numata, K., Wu, S., and Fahey, M.: The challenges of
measuring methane from space with a LIDAR, CEAS Space
J., 11, 475–483, https://doi.org/10.1007/s12567-019-00274-8,
2019.
Rodgers, C. D.: Inverse Methods for Atmospheric Sounding: Theory and Practice, World Scientific, River Edge, USA, ISBN 978-
9810227401, 2000.
Rohrschneider, R. R., Wofsy, S, Franklin, J. E., Benmergui, J., Soto,
J., and Davis, S. B.: The MethaneSAT Mission, 35th Annual
Small Satellite Conference Proceedings, virtual, 7–12 August
2021, SSC21-II-05, 1–7, 2021.
Sadavarte, P., Pandey, S., Maasakkers, J. D., Lorente, A., Gon,
H. D., van der Houweling, S., and Aben, I.: Methane emissions from super-emitting coal mines in Australia quantified using TROPOMI satellite observations, Environ. Sci. Technol., 55,
16573–16580, https://doi.org/10.1021/acs.est.1c03976, 2021.
Sánchez-García, E., Gorroño, J., Irakulis-Loitxate, I., Varon,
D. J., and Guanter, L.: Mapping methane plumes at very
high spatial resolution with the WorldView-3 satellite, Atmos.
Meas. Tech., 15, 1657–1674, https://doi.org/10.5194/amt-15-
1657-2022, 2022.
Saunois, M., Stavert, A. R., Poulter, B., Bousquet, P., Canadell, J.
G., Jackson, R. B., Raymond, P. A., Dlugokencky, E. J., Houweling, S., Patra, P. K., Ciais, P., Arora, V. K., Bastviken, D., Bergamaschi, P., Blake, D. R., Brailsford, G., Bruhwiler, L., Carlson, K. M., Carrol, M., Castaldi, S., Chandra, N., Crevoisier, C.,
Crill, P. M., Covey, K., Curry, C. L., Etiope, G., Frankenberg,
C., Gedney, N., Hegglin, M. I., Höglund-Isaksson, L., Hugelius,
G., Ishizawa, M., Ito, A., Janssens-Maenhout, G., Jensen, K.
M., Joos, F., Kleinen, T., Krummel, P. B., Langenfelds, R. L.,
Laruelle, G. G., Liu, L., Machida, T., Maksyutov, S., McDonald, K. C., McNorton, J., Miller, P. A., Melton, J. R., Morino,
I., Müller, J., Murguia-Flores, F., Naik, V., Niwa, Y., Noce, S.,
O’Doherty, S., Parker, R. J., Peng, C., Peng, S., Peters, G. P.,
Prigent, C., Prinn, R., Ramonet, M., Regnier, P., Riley, W. J.,
Rosentreter, J. A., Segers, A., Simpson, I. J., Shi, H., Smith, S.
J., Steele, L. P., Thornton, B. F., Tian, H., Tohjima, Y., Tubiello,
F. N., Tsuruta, A., Viovy, N., Voulgarakis, A., Weber, T. S.,
van Weele, M., van der Werf, G. R., Weiss, R. F., Worthy, D.,
Wunch, D., Yin, Y., Yoshida, Y., Zhang, W., Zhang, Z., Zhao,
Y., Zheng, B., Zhu, Q., Zhu, Q., and Zhuang, Q.: The Global
Methane Budget 2000–2017, Earth Syst. Sci. Data, 12, 1561–
1623, https://doi.org/10.5194/essd-12-1561-2020, 2020.
Scarfutto, R. D. M., van der Werf, H., Bakker, W. H., van der Meer,
F., and de Souza Filho, C. R.: An evaluation of airborne SWIR
imaging spectrometers for CH4 mapping: Implications of band
positioning, spectral sampling and noise, Int. J. Appl. Earth Obs.,
94, 102233, https://doi.org/10.1016/j.jag.2020.102233, 2021.
Scarpelli, T. R., Jacob, D. J., Maasakkers, J. D., Sulprizio, M. P.,
Sheng, J.-X., Rose, K., Romeo, L., Worden, J. R., and JanssensMaenhout, G.: A global gridded (0.1
◦ × 0.1
◦
) inventory of
methane emissions from oil, gas, and coal exploitation based
on national reports to the United Nations Framework Convention on Climate Change, Earth Syst. Sci. Data, 12, 563–575,
https://doi.org/10.5194/essd-12-563-2020, 2020.
Schneising, O., Buchwitz, M., Reuter, M., Bovensmann, H., Burrows, J. P., Borsdorff, T., Deutscher, N. M., Feist, D. G., Griffith, D. W. T., Hase, F., Hermans, C., Iraci, L. T., Kivi, R.,
Landgraf, J., Morino, I., Notholt, J., Petri, C., Pollard, D. F.,
Roche, S., Shiomi, K., Strong, K., Sussmann, R., Velazco, V. A.,
Warneke, T., and Wunch, D.: A scientific algorithm to simultaneously retrieve carbon monoxide and methane from TROPOMI
onboard Sentinel-5 Precursor, Atmos. Meas. Tech., 12, 6771–
6802, https://doi.org/10.5194/amt-12-6771-2019, 2019.
Seinfeld, J. H and Pandis, S.: Atmospheric Chemistry and Physics,
3rd edn., Wiley, ISBN 9781118947401, 2016.
Shen, L., Zavala-Araiza, D., Gautam, R., Omara, M., Scarpelli,
T., Sheng, J., Sulprizio, M. P., Zhuang, J., Zhang, Y.,
Lu, X., Hamburg, S. P., and Jacob, D. J.: Unravelling
a large methane emission discrepancy in Mexico using
satellite observations, Remote Sens. Environ., 260, 112461,
https://doi.org/10.1016/j.rse.2021.112461, 2021.
Shen, L., Gautam, R., Omara, M., Zavala-Araiza, D., Maasakkers,
J., Scarpelli, T., Lorente, A., Lyon, D., Sheng, J., Varon, D.,
Nesser, H., Qu, Z., Lu, X., Sulprizio, M., Hamburg, S., and Jacob, D.: Satellite quantification of oil and natural gas methane
emissions in the US and Canada including contributions from
individual basins, Atmos. Chem. Phys. Discuss. [preprint],
https://doi.org/10.5194/acp-2022-155, in review, 2022.
Sherwin, A. D., Rutherford, J. S., Chen, Y., Kort, E. A., Jackson, R. B., and Brandt, A. R.: Single-blind validation of spacebased point-source methane emissions detection and quantification, EarthArXiv, https://doi.org/10.31223/X5DH09, 2022.
Sierk, B., Bézy, J.-L., L¨scher, A., and Meijer, Y.: The European
CO2 Monitoring Mission: observing anthropogenic greenhouse
gas emissions from space, International Conference on Space
Optics – ICSO 2018, Platanias, Greece, 12 July 2019, Proc. SPIE
11180, https://doi.org/10.1117/12.2535941, 2019.
Suto, H., Kataoka, F., Kikuchi, N., Knuteson, R. O., Butz, A., Haun,
M., Buijs, H., Shiomi, K., Imai, H., and Kuze, A.: Thermal and
near-infrared sensor for carbon observation Fourier transform
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
D. J. Jacob et al.: Quantifying methane emissions using satellites 9645
spectrometer-2 (TANSO-FTS-2) on the Greenhouse gases Observing SATellite-2 (GOSAT-2) during its first year in orbit, Atmos. Meas. Tech., 14, 2013–2039, https://doi.org/10.5194/amt14-2013-2021, 2021.
Thompson, D. R., Leifer, I., Bovensmann, H., Eastwood, M.,
Fladeland, M., Frankenberg, C., Gerilowski, K., Green, R. O.,
Kratwurst, S., Krings, T., Luna, B., and Thorpe, A. K.: Real-time
remote detection and measurement for airborne imaging spectroscopy: a case study with methane, Atmos. Meas. Tech., 8,
4383–4397, https://doi.org/10.5194/amt-8-4383-2015, 2015.
Thompson, D. R., Thorpe, A. K., Frankenberg, C., Green, R. O.,
Duren, R., Guanter, L., Hollstein, A., Middleton, E., Ong, L.,
and Ungar, S.: Space-based remote imaging spectroscopy of the
Aliso Canyon CH4 superemitter, Geophys. Res. Lett., 43, 6571–
6578, https://doi.org/10.1002/2016GL069079, 2016.
Thorpe, A. K., Frankenberg, C., and Roberts, D. A.: Retrieval techniques for airborne imaging of methane concentrations using high spatial and moderate spectral resolution:
application to AVIRIS, Atmos. Meas. Tech., 7, 491–506,
https://doi.org/10.5194/amt-7-491-2014, 2014.
Thorpe, A. K., Frankenberg, C., Thompson, D. R., Duren, R.
M., Aubrey, A. D., Bue, B. D., Green, R. O., Gerilowski, K.,
Krings, T., Borchardt, J., Kort, E. A., Sweeney, C., Conley,
S., Roberts, D. A., and Dennison, P. E.: Airborne DOAS retrievals of methane, carbon dioxide, and water vapor concentrations at high spatial resolution: application to AVIRIS-NG, Atmos. Meas. Tech., 10, 3833–3850, https://doi.org/10.5194/amt10-3833-2017, 2017.
Tu, Q., Hase, F., Schneider, M., García, O., Blumenstock, T., Borsdorff, T., Frey, M., Khosrawi, F., Lorente, A., Alberti, C., Bustos, J. J., Butz, A., Carreño, V., Cuevas, E., Curcoll, R., Diekmann, C. J., Dubravica, D., Ertl, B., Estruch, C., León-Luis, S.
F., Marrero, C., Morgui, J.-A., Ramos, R., Scharun, C., Schneider, C., Sepúlveda, E., Toledano, C., and Torres, C.: Quantification of CH4 emissions from waste disposal sites near the city
of Madrid using ground- and space-based observations of COCCON, TROPOMI and IASI, Atmos. Chem. Phys., 22, 295–317,
https://doi.org/10.5194/acp-22-295-2022, 2022.
Turner, A. J., Jacob, D. J., Wecht, K. J., Maasakkers, J. D., Lundgren, E., Andrews, A. E., Biraud, S. C., Boesch, H., Bowman, K.
W., Deutscher, N. M., Dubey, M. K., Griffith, D. W. T., Hase,
F., Kuze, A., Notholt, J., Ohyama, H., Parker, R., Payne, V.
H., Sussmann, R., Sweeney, C., Velazco, V. A., Warneke, T.,
Wennberg, P. O., and Wunch, D.: Estimating global and North
American methane emissions with high spatial resolution using GOSAT satellite data, Atmos. Chem. Phys., 15, 7049–7069,
https://doi.org/10.5194/acp-15-7049-2015, 2015.
United Nations Environment Programme: An Eye on Methane: International Methane Emissions Observatory, Nairobi, ISBN 978-
92-807-3893-3, 2021.
Van Damme, M., Clarisse, L., Whitburn, M., Hadji-Lazaro, J., Hurtmans, D., Clerbaux, C., and Coheur, P.-F.: Industrial and agricultural ammonia point sources exposed, Nature, 564, 99–103,
https://doi.org/10.1038/s41586-018-0747-1, 2018.
Varon, D. J., Jacob, D. J., McKeever, J., Jervis, D., Durak,
B. O. A., Xia, Y., and Huang, Y.: Quantifying methane
point sources from fine-scale satellite observations of atmospheric methane plumes, Atmos. Meas. Tech., 11, 5673–5686,
https://doi.org/10.5194/amt-11-5673-2018, 2018.
Varon, D. J., McKeever, J., Jervis, D., Maasakkers, J. D., Pandey,
S., Houweling, S., Aben, I., Scarpelli, T. R., and Jacob, D. J.:
Satellite discovery of anomalously large methane point sources
from oil/gas production, Geophys. Res. Lett., 46, 13507–13516,
https://doi.org/10.1029/2019GL083798, 2019.
Varon, D. J., Jacob, D. J., Jervis, D., and McKeever, J.: Quantifying time-averaged methane emissions from individual coal mine
vents with GHGSat-D satellite observations, Environ. Sci. Technol., 54, 10246–10253, https://doi.org/10.1021/acs.est.0c01213,
2020.
Varon, D. J., Jervis, D., McKeever, J., Spence, I., Gains,
D., and Jacob, D. J.: High-frequency monitoring of anomalous methane point sources with multispectral Sentinel-2
satellite observations, Atmos. Meas. Tech., 14, 2771–2785,
https://doi.org/10.5194/amt-14-2771-2021, 2021.
Varon, D. J., Jacob, D. J., Sulprizio, M., Estrada, L., Downs, W. B.,
Shen, L., Hancock, S. E., Nesser, H., Qu, Z., Penn, E., Chen,
Z., Lu, X., Lorente, A., Tewari, A., and Randles, C. A.: Integrated Methane Inversion (IMI 1.0): A user-friendly, cloudbased facility for inferring high-resolution methane emissions
from TROPOMI satellite observations, Geosci. Model Dev. Discuss. [preprint], https://doi.org/10.5194/gmd-2022-45, in review,
2022.
Vaughn, T. L., Bell. C. S., Pickering, C. K, and Nummedal, D.:
Temporal variability largely explains top-down/bottom-up difference in methane emission estimates from a natural gas production region, P. Natl. Acad. Sci. USA, 115, 11712–11717,
https://doi.org/10.1073/pnas.1805687115, 2018.
Wang, F., Maksyutov, S., Tsuruta, A., Janardanan, R., Ito,
A., Sasakawa, M., Machida, T., Morino, I., Yoshida, Y.,
Kaiser, J. W., Janssens-Maenhout, G., Dlugokencky, E. J.,
Mammarella, I., Lavric, J. V., and Matsunaga, T.: Methane
emission estimates by the global high-resolution inverse
model using national inventories, Remote Sens., 11, 2489,
https://doi.org/10.3390/rs11212489, 2019.
Wecht, K. J., Jacob, D. J., Frankenberg, C., Jiang, Z., and
Blake, D. R.: Mapping of North America methane emissions with high spatial resolution by inversion of SCIAMACHY satellite data, J. Geophys. Res., 119, 7741–7756,
https://doi.org/10.1002/2014JD021551, 2014.
Western, L. M., Ramsden, A. E., Ganesan, A. L., Boesch, H.,
Parker, R. J., Scarpelli, T. R., Tunnicliffe, R. L., and Rigby, M.:
Estimates of North African methane emissions from 2010 to
2017 using GOSAT observations, Environ. Sci. Technol. Lett.,
8, 626–632, https://doi.org/10.1021/acs.estlett.1c00327, 2021.
White, W. H., Anderson, J. A., Blumenthal, D. L., Husar, R. B.,
Gillani, N. V., Husar, J. D., and Wilson, W. E.: Formation and
transport of secondary air pollutants: ozone and aerosols in the
St. Louis urban plume, Science, 194, 187–189, 1976.
Worden, J., Wecht, K., Frankenberg, C., Alvarado, M., Bowman,
K., Kort, E., Kulawik, S., Lee, M., Payne, V., and Worden,
H.: CH4 and CO distributions over tropical fires during October 2006 as observed by the Aura TES satellite instrument and
modeled by GEOS-Chem, Atmos. Chem. Phys., 13, 3679–3692,
https://doi.org/10.5194/acp-13-3679-2013, 2013.
Worden, J. R., Turner, A. J., Bloom, A., Kulawik, S. S., Liu,
J., Lee, M., Weidner, R., Bowman, K., Frankenberg, C.,
Parker, R., and Payne, V. H.: Quantifying lower tropospheric
methane concentrations using GOSAT near-IR and TES therhttps://doi.org/10.5194/acp-22-9617-2022 Atmos. Chem. Phys., 22, 9617–9646, 2022
9646 D. J. Jacob et al.: Quantifying methane emissions using satellites
mal IR measurements, Atmos. Meas. Tech., 8, 3433–3445,
https://doi.org/10.5194/amt-8-3433-2015, 2015.
Worden, J. R., Cusworth, D. H., Qu, Z., Yin, Y., Zhang, Y., Bloom,
A. A., Ma, S., Byrne, B. K., Scarpelli, T., Maasakkers, J. D.,
Crisp, D., Duren, R., and Jacob, D. J.: The 2019 methane budget and uncertainties at 1◦
resolution and each country through
Bayesian integration Of GOSAT total column methane data and a
priori inventory estimates, Atmos. Chem. Phys., 22, 6811–6841,
https://doi.org/10.5194/acp-22-6811-2022, 2022.
Wunch, D., Toon, G. C., Blavier, J.-F. L., Washenfelder, R. A.,
Notholt, J., Connor, B. J., Griffith, D. W. T., Sherlock,V., and
Wennberg, P. O.: The Total Carbon Column Observing Network,
Philos. T. R. Soc. A, 369, 2087–2112, 2011.
Yang, S., Lan, X., Talbot, R., and Liu, T.: Characterizing anthropogenic methane sources in the Houston and
Barnett Shale areas of Texas using the isotopic signature δ
13C in CH4, Sci. Total. Environ., 696, 133856,
https://doi.org/10.1016/j.scitotenv.2019.133856, 2019.
Yin, Y., Chevallier, F., Ciais, P., Bousquet, P., Saunois, M., Zheng,
B., Worden, J., Bloom, A. A., Parker, R. J., Jacob, D. J., Dlugokencky, E. J., and Frankenberg, C.: Accelerating methane
growth rate from 2010 to 2017: leading contributions from the
tropics and East Asia, Atmos. Chem. Phys., 21, 12631–12647,
https://doi.org/10.5194/acp-21-12631-2021, 2021.
Yu, X., Millet, D. B., and Henze, D. K.: How well can inverse
analyses of high-resolution satellite data resolve heterogeneous
methane fluxes? Observing system simulation experiments with
the GEOS-Chem adjoint model (v35), Geosci. Model Dev., 14,
7775–7793, https://doi.org/10.5194/gmd-14-7775-2021, 2021.
Yuan, B., Kaser, L., Karl, T., Graus, M., Peischl, J., Campos,
T. L., Shertz, S., Apel, E. C., Hornbrook, R. S., Hills, A.,
Gilman, J. B., Lerner, B. M., Warneke, C., Flocke, F. M.,
Ryerson, T. B., Guenther, A. B., and de Gouw, J. A.: Airborne flux measurements of methane and volatile organic compounds over the Haynesville and Marcellus shale gas production regions, J. Geophys. Res.-Atmos., 120, 6271–6289,
https://doi.org/10.1002/2015JD023242, 2015.
Zhang, Y., Gautam, R., Pandey, S., Omara, M., Maasakkers,
J. D., Sadavarte, P., Lyon, D., Nesser, H., Sulprizio, M. P.,
Varon, D. J., Zhang, R., Houweling, S., Zavala-Araiza, D., Alvarez, R. A., Lorente, A., Hamburg, S. P., Aben, I., and Jacob, D. J.: Quantifying methane emissions from the largest oilproducing basin in the United States from space, Sci. Adv., 6, 17,
https://doi.org/10.1126/sciadv.aaz5120, 2020.
Zhang, Y., Jacob, D. J., Lu, X., Maasakkers, J. D., Scarpelli, T. R.,
Sheng, J.-X., Shen, L., Qu, Z., Sulprizio, M. P., Chang, J., Bloom,
A. A., Ma, S., Worden, J., Parker, R. J., and Boesch, H.: Attribution of the accelerating increase in atmospheric methane during 2010–2018 by inverse analysis of GOSAT observations, Atmos. Chem. Phys., 21, 3643–3666, https://doi.org/10.5194/acp21-3643-2021, 2021.
Zimmerle, D., Duggan, G., Vaughn, T., Bell, C., Lute, C., Bennett, K., Kimura, Y., Cardoso-Saldaña, F. J., and Allen, D.
T.: Modeling air emissions from complex facilities at detailed temporal and spatial resolution: The Methane Emission
Estimation Tool (MEET), Sci. Total Environ., 824, 153653,
https://doi.org/10.1016/j.scitotenv.2022.153653, 2022.
Atmos. Chem. Phys., 22, 9617–9646, 2022 https://doi.org/10.5194/acp-22-9617-2022
"""

In [20]:
config = StoryMDXBuilderConfig(api_key=os.environ.get("OPENAI_API_KEY"))
story_mdx_builder = StoryMDXBuilderComponent(config)
res = await story_mdx_builder.process(query, datasets, raw_contents)

In [21]:
res

'<Block>\n<Prose>\n### The Unwritten Story of Our World\nIn a world filled with untold stories, every dataset holds a piece of the puzzle that shapes our understanding of the environment, society, and the intricate web of life that surrounds us. Each dataset is not just a collection of numbers; it is a narrative waiting to be uncovered.\n\nImagine a vast landscape where data flows like rivers, connecting communities, ecosystems, and economies. From the air we breathe to the food we eat, every aspect of our lives is influenced by the information we gather and analyze. \n\n#### The Power of Data\nData is the lifeblood of decision-making. It empowers scientists, policymakers, and activists to make informed choices that can lead to a sustainable future. For instance, datasets on air quality can reveal trends in pollution levels, guiding efforts to improve public health. Similarly, agricultural data can help farmers optimize their practices, ensuring food security for future generations.\n\